# CIFAR-10 Adversarial Attack and Defense Evaluation

This notebook is the reproducible evaluation entry point for the
project. It operates on raw CIFAR-10 images in `[0, 1]`; input
normalization remains inside the model.

The notebook supports two modes:

- `smoke`: small deterministic subsets for structural validation.
- `final`: project-scale evaluation and report generation.

Main responsibilities:

1. Restore checkpoints from Google Drive.
2. Load the clean, PGD-AT, and Gaussian-fine-tuned classifiers.
3. Load attack and defense implementations.
4. Run deterministic smoke checks before expensive evaluation.
5. Produce the final metrics table and project figures.

## 1. Runtime and repository setup

This section mounts Google Drive and clones the repository when
necessary. If the repository already exists and is clean, it is
updated with a fast-forward-only pull.

In [1]:
from pathlib import Path
import os
import subprocess

from google.colab import drive

REPOSITORY_URL = (
    "https://github.com/MostafaOmidi17/"
    "AI-Project-Adversarial-Attack-Defense.git"
)

PROJECT_ROOT = Path(
    "/content/AI-Project-Adversarial-Attack-Defense"
)

drive.mount("/content/drive")

if not (PROJECT_ROOT / ".git").is_dir():
    subprocess.run(
        [
            "git",
            "clone",
            REPOSITORY_URL,
            str(PROJECT_ROOT),
        ],
        check=True,
    )
else:
    repository_status = subprocess.run(
        [
            "git",
            "-C",
            str(PROJECT_ROOT),
            "status",
            "--porcelain",
        ],
        check=True,
        text=True,
        capture_output=True,
    ).stdout.strip()

    if repository_status:
        print(
            "Repository contains local changes; "
            "automatic pull was skipped."
        )
    else:
        subprocess.run(
            [
                "git",
                "-C",
                str(PROJECT_ROOT),
                "pull",
                "--ff-only",
            ],
            check=True,
        )

os.chdir(PROJECT_ROOT)

current_commit = subprocess.run(
    [
        "git",
        "rev-parse",
        "--short",
        "HEAD",
    ],
    check=True,
    text=True,
    capture_output=True,
).stdout.strip()

print("Project root:", PROJECT_ROOT)
print("Git commit:", current_commit)

Mounted at /content/drive
Project root: /content/AI-Project-Adversarial-Attack-Defense
Git commit: 0c22817


## 2. Dependencies

Requirements are installed inside the current Colab runtime.
Diffusion-related dependencies are listed explicitly because the
purification model is loaded from Hugging Face.

In [2]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(PROJECT_ROOT / "requirements.txt"),
    ],
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "diffusers",
        "huggingface_hub",
        "safetensors",
        "accelerate",
    ],
    check=True,
)

print("Dependencies installed.")

Dependencies installed.


## 3. Reproducibility, paths, and evaluation mode

`smoke` mode must pass before `final` mode is enabled.

In [3]:
import hashlib
import json
import random
import shutil
import time

import numpy as np
import pandas as pd
import torch
import torchvision

from torch.utils.data import DataLoader, Subset
from torchvision import transforms

SEED = 42
RUN_MODE = "smoke"

if RUN_MODE not in {"smoke", "final"}:
    raise ValueError(
        "RUN_MODE must be 'smoke' or 'final'."
    )

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

DRIVE_PROJECT_DIR = Path(
    "/content/drive/MyDrive/"
    "AI-Project-Adversarial-Attack-Defense"
)

DRIVE_CHECKPOINT_DIR = (
    DRIVE_PROJECT_DIR / "checkpoints"
)

DRIVE_RESULTS_DIR = (
    DRIVE_PROJECT_DIR / "results"
)

for directory in (
    CHECKPOINT_DIR,
    DATA_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    DRIVE_CHECKPOINT_DIR,
    DRIVE_RESULTS_DIR,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

MODE_CONFIG = {
    "smoke": {
        "general_samples": 64,
        "expensive_attack_samples": 4,
        "diffusion_samples": 2,
    },
    "final": {
        "general_samples": 10000,
        "expensive_attack_samples": 1000,
        "diffusion_samples": 200,
    },
}

active_config = MODE_CONFIG[RUN_MODE]

print("Run mode:", RUN_MODE)
print("Device:", device)
print("Configuration:", active_config)

Run mode: smoke
Device: cuda:0
Configuration: {'general_samples': 64, 'expensive_attack_samples': 4, 'diffusion_samples': 2}


## 4. Restore and verify checkpoints

Checkpoints are not stored in Git. They are restored from Google
Drive and verified using known SHA-256 hashes.

In [4]:
EXPECTED_CHECKPOINT_HASHES = {
    "resnet20_clean_best.pt":
        "8874dc56b8fd452aba4cba76774b3e3cab6e16f3740ab6437c3e924743f5c9b0",

    "resnet20_pgd_at_eps8_best.pt":
        "e63b5f7f483453203bb0224d6967b9b88ae28f68e72e1d8a99bfc66738e94210",

    "resnet20_pgd_at_eps8_last.pt":
        "28e0ced68d513ee320d95d007fd5953d2b98a62b626f7988f0ec4d157fd78872",

    "resnet20_rand_smooth_sigma0p25_best.pt":
        "c5f7c857c1c43d4df5b3376a404c5f7faa4e554ed6d6867291e9cda8b2dbc238",

    "resnet20_rand_smooth_sigma0p25_last.pt":
        "92506d2cb180574e4eab5b681e8df41c5309e1fc60506d8590313b0c717fc324",
}


def calculate_sha256(path: Path) -> str:
    with path.open("rb") as checkpoint_file:
        return hashlib.file_digest(
            checkpoint_file,
            "sha256",
        ).hexdigest()


for checkpoint_name, expected_hash in (
    EXPECTED_CHECKPOINT_HASHES.items()
):
    drive_path = (
        DRIVE_CHECKPOINT_DIR / checkpoint_name
    )

    local_path = (
        CHECKPOINT_DIR / checkpoint_name
    )

    if not drive_path.is_file():
        raise FileNotFoundError(
            f"Drive checkpoint is missing: {drive_path}"
        )

    if (
        not local_path.is_file()
        or calculate_sha256(local_path)
        != expected_hash
    ):
        shutil.copy2(
            drive_path,
            local_path,
        )

    actual_hash = calculate_sha256(
        local_path
    )

    if actual_hash != expected_hash:
        raise RuntimeError(
            f"Checkpoint hash mismatch: "
            f"{checkpoint_name}"
        )

    print(
        f"VERIFIED: {checkpoint_name} "
        f"({local_path.stat().st_size:,} bytes)"
    )

print(
    "\nALL CHECKPOINTS RESTORED AND VERIFIED."
)

VERIFIED: resnet20_clean_best.pt (2,248,465 bytes)
VERIFIED: resnet20_pgd_at_eps8_best.pt (2,249,723 bytes)
VERIFIED: resnet20_pgd_at_eps8_last.pt (2,249,723 bytes)
VERIFIED: resnet20_rand_smooth_sigma0p25_best.pt (2,249,729 bytes)
VERIFIED: resnet20_rand_smooth_sigma0p25_last.pt (2,249,729 bytes)

ALL CHECKPOINTS RESTORED AND VERIFIED.


## 5. CIFAR-10 test data

The external transform is only `ToTensor()`. Normalization is
performed by the model. Deterministic nested subsets are used so
that expensive methods evaluate the same initial test examples.

In [5]:
test_dataset = torchvision.datasets.CIFAR10(
    root=str(DATA_DIR),
    train=False,
    download=True,
    transform=transforms.ToTensor(),
)

subset_generator = (
    torch.Generator().manual_seed(SEED)
)

fixed_permutation = torch.randperm(
    len(test_dataset),
    generator=subset_generator,
).tolist()

general_sample_count = min(
    active_config["general_samples"],
    len(test_dataset),
)

expensive_sample_count = min(
    active_config[
        "expensive_attack_samples"
    ],
    general_sample_count,
)

diffusion_sample_count = min(
    active_config["diffusion_samples"],
    general_sample_count,
)

general_indices = fixed_permutation[
    :general_sample_count
]

expensive_indices = fixed_permutation[
    :expensive_sample_count
]

diffusion_indices = fixed_permutation[
    :diffusion_sample_count
]

general_dataset = Subset(
    test_dataset,
    general_indices,
)

expensive_dataset = Subset(
    test_dataset,
    expensive_indices,
)

diffusion_dataset = Subset(
    test_dataset,
    diffusion_indices,
)

general_loader = DataLoader(
    general_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

expensive_loader = DataLoader(
    expensive_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

diffusion_loader = DataLoader(
    diffusion_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

sample_images, sample_labels = next(
    iter(general_loader)
)

assert sample_images.shape[1:] == (
    3,
    32,
    32,
)

assert sample_images.min().item() >= 0.0
assert sample_images.max().item() <= 1.0

print("Full test size:", len(test_dataset))
print("General subset:", len(general_dataset))
print(
    "Expensive-attack subset:",
    len(expensive_dataset),
)
print(
    "Diffusion subset:",
    len(diffusion_dataset),
)
print("CIFAR-10 DATA SETUP PASSED.")

100%|██████████| 170M/170M [14:21<00:00, 198kB/s]


Full test size: 10000
General subset: 64
Expensive-attack subset: 4
Diffusion subset: 2
CIFAR-10 DATA SETUP PASSED.


## 6. Load project modules and classifier checkpoints

In [6]:
from src.models import (
    build_normalized_resnet20,
)

from src.attacks.fgsm import fgsm_attack
from src.attacks.pgd import pgd_attack
from src.attacks.deepfool import deepfool_attack
from src.attacks.cw_l2 import cw_l2_attack

from src.defenses.randomized_smoothing import (
    predict_smoothed,
)

from src.defenses.diffusion_purification import (
    DEFAULT_DIFFUSION_MODEL_ID,
    load_diffusion_components,
    purify_images,
)

clean_model = build_normalized_resnet20(
    checkpoint_path=(
        CHECKPOINT_DIR
        / "resnet20_clean_best.pt"
    ),
    device=device,
    eval_mode=True,
)

pgd_at_model = build_normalized_resnet20(
    checkpoint_path=(
        CHECKPOINT_DIR
        / "resnet20_pgd_at_eps8_best.pt"
    ),
    device=device,
    eval_mode=True,
)

smoothing_model = build_normalized_resnet20(
    checkpoint_path=(
        CHECKPOINT_DIR
        / "resnet20_rand_smooth_sigma0p25_best.pt"
    ),
    device=device,
    eval_mode=True,
)

classifier_models = {
    "none": clean_model,
    "pgd_at": pgd_at_model,
    "rand_smooth": smoothing_model,
}

for defense_id, model in (
    classifier_models.items()
):
    assert (
        next(model.parameters()).device
        == device
    )

    print(
        f"Loaded {defense_id} on "
        f"{next(model.parameters()).device}"
    )

print("CLASSIFIER CHECKPOINT LOADING PASSED.")

Loaded none on cuda:0
Loaded pgd_at on cuda:0
Loaded rand_smooth on cuda:0
CLASSIFIER CHECKPOINT LOADING PASSED.


## 7. Structural smoke test

This test performs very small attack calls. It does not write
final metrics and does not load the diffusion model yet.

In [7]:
smoke_images = sample_images[:2].to(device)
smoke_labels = sample_labels[:2].to(device)

for defense_id, model in (
    classifier_models.items()
):
    with torch.no_grad():
        logits = model(smoke_images)

    assert logits.shape == (2, 10)
    assert torch.isfinite(logits).all()

    print(
        f"{defense_id} forward PASSED: "
        f"{tuple(logits.shape)}"
    )

smoke_fgsm = fgsm_attack(
    model=clean_model,
    images=smoke_images,
    labels=smoke_labels,
    epsilon=8 / 255,
)

smoke_pgd = pgd_attack(
    model=clean_model,
    images=smoke_images,
    labels=smoke_labels,
    epsilon=8 / 255,
    alpha=2 / 255,
    steps=2,
    random_start=True,
    restarts=1,
)

smoke_deepfool = deepfool_attack(
    model=clean_model,
    images=smoke_images,
    labels=smoke_labels,
    max_steps=1,
    overshoot=0.02,
    num_classes=10,
)

smoke_cw = cw_l2_attack(
    model=clean_model,
    images=smoke_images,
    labels=smoke_labels,
    c=1.0,
    kappa=0.0,
    learning_rate=0.01,
    steps=2,
)

for attack_id, adversarial_images in {
    "fgsm": smoke_fgsm,
    "pgd": smoke_pgd,
    "deepfool": smoke_deepfool,
    "cw_l2": smoke_cw,
}.items():
    assert (
        adversarial_images.shape
        == smoke_images.shape
    )

    assert (
        adversarial_images.min().item()
        >= 0.0
    )

    assert (
        adversarial_images.max().item()
        <= 1.0
    )

    print(
        f"{attack_id} structural call PASSED."
    )

smoothing_generator = torch.Generator(
    device=device,
).manual_seed(SEED)

(
    smoke_smoothed_predictions,
    smoke_vote_counts,
) = predict_smoothed(
    model=smoothing_model,
    images=smoke_images,
    sigma=0.25,
    num_samples=7,
    noise_batch_size=3,
    num_classes=10,
    clip_noise=True,
    generator=smoothing_generator,
)

assert (
    smoke_smoothed_predictions.shape
    == (2,)
)

assert smoke_vote_counts.shape == (
    2,
    10,
)

assert torch.all(
    smoke_vote_counts.sum(dim=1) == 7
)

print(
    "Randomized smoothing structural call PASSED."
)

print(
    "\nEVALUATION NOTEBOOK PHASE-1 "
    "SMOKE TEST PASSED."
)

none forward PASSED: (2, 10)
pgd_at forward PASSED: (2, 10)
rand_smooth forward PASSED: (2, 10)
fgsm structural call PASSED.
pgd structural call PASSED.
deepfool structural call PASSED.
cw_l2 structural call PASSED.
Randomized smoothing structural call PASSED.

EVALUATION NOTEBOOK PHASE-1 SMOKE TEST PASSED.


## Next phase

After this setup passes in a fresh Colab runtime, the notebook
will be extended with:

- contract-compliant metric aggregation;
- clean, FGSM, PGD, DeepFool, and C&W evaluation;
- randomized-smoothing defended predictions;
- diffusion purification on the fixed subset;
- CSV/config persistence and final visualization generation.

In [8]:
from pathlib import Path
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import gc
import json
import math
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from src.attacks.deepfool import deepfool_attack
from src.attacks.cw_l2 import cw_l2_attack


# --------------------------------------------------
# پیدا کردن مسیر پروژه
# --------------------------------------------------

PROJECT_ROOT = Path(
    globals().get(
        "PROJECT_ROOT",
        "/content/AI-Project-Adversarial-Attack-Defense",
    )
)

assert PROJECT_ROOT.is_dir(), PROJECT_ROOT


# --------------------------------------------------
# پیدا کردن دیکشنری مدل‌هایی که در Phase 1 لود شدند
# --------------------------------------------------

preferred_model_names = (
    "CLASSIFIERS",
    "classifiers",
    "models_by_defense",
    "loaded_models",
    "models",
)

CLASSIFIERS = None
CLASSIFIER_VARIABLE_NAME = None

for variable_name in preferred_model_names:
    value = globals().get(variable_name)

    if (
        isinstance(value, dict)
        and {"none", "pgd_at", "rand_smooth"}.issubset(value)
        and all(
            isinstance(value[key], nn.Module)
            for key in ("none", "pgd_at", "rand_smooth")
        )
    ):
        CLASSIFIERS = value
        CLASSIFIER_VARIABLE_NAME = variable_name
        break

if CLASSIFIERS is None:
    for variable_name, value in list(globals().items()):
        if (
            isinstance(value, dict)
            and {"none", "pgd_at", "rand_smooth"}.issubset(value)
            and all(
                isinstance(value[key], nn.Module)
                for key in ("none", "pgd_at", "rand_smooth")
            )
        ):
            CLASSIFIERS = value
            CLASSIFIER_VARIABLE_NAME = variable_name
            break

assert CLASSIFIERS is not None, (
    "Could not find the Phase-1 classifier dictionary."
)

DEVICE = next(
    CLASSIFIERS["none"].parameters()
).device

for model in CLASSIFIERS.values():
    model.eval()


# --------------------------------------------------
# پیدا کردن دیتاست تست
# --------------------------------------------------

TEST_DATASET = None

for variable_name in (
    "test_dataset",
    "cifar10_test",
    "test_data",
):
    candidate = globals().get(variable_name)

    if (
        candidate is not None
        and hasattr(candidate, "__len__")
        and len(candidate) == 10_000
    ):
        TEST_DATASET = candidate
        break

if TEST_DATASET is None:
    TEST_DATASET = datasets.CIFAR10(
        root="/content/data",
        train=False,
        download=True,
        transform=transforms.ToTensor(),
    )

assert len(TEST_DATASET) == 10_000


# --------------------------------------------------
# subset ثابت فقط برای اندازه‌گیری زمان
# --------------------------------------------------

SEED = 42
CALIBRATION_SAMPLES = 64

subset_generator = torch.Generator()
subset_generator.manual_seed(SEED)

CALIBRATION_INDICES = torch.randperm(
    len(TEST_DATASET),
    generator=subset_generator,
)[:CALIBRATION_SAMPLES].tolist()


DRIVE_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/"
    "AI-Project-Adversarial-Attack-Defense"
)

DRIVE_RESULTS_DIR = DRIVE_PROJECT_ROOT / "results"
DRIVE_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Classifier dictionary:", CLASSIFIER_VARIABLE_NAME)
print("Device:", DEVICE)
print("Test samples:", len(TEST_DATASET))
print("Calibration samples:", len(CALIBRATION_INDICES))
print("First 10 indices:", CALIBRATION_INDICES[:10])
print("Drive results:", DRIVE_RESULTS_DIR)

print("\nPHASE-2A CALIBRATION SETUP PASSED.")

Classifier dictionary: classifier_models
Device: cuda:0
Test samples: 10000
Calibration samples: 64
First 10 indices: [7542, 8214, 3698, 2841, 8086, 2550, 2282, 7599, 1462, 2026]
Drive results: /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results

PHASE-2A CALIBRATION SETUP PASSED.


In [9]:
CHECKPOINT_NAMES = {
    "none": "resnet20_clean_best.pt",
    "pgd_at": "resnet20_pgd_at_eps8_best.pt",
    "rand_smooth": (
        "resnet20_rand_smooth_sigma0p25_best.pt"
    ),
}


def synchronize_device():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)


def evaluate_expensive_attack(
    defense_id,
    attack_id,
    indices,
):
    if defense_id not in ("none", "pgd_at"):
        raise ValueError(
            "This Phase-2A function currently evaluates "
            "none and pgd_at only."
        )

    model = CLASSIFIERS[defense_id]
    model.eval()

    if attack_id == "deepfool":
        batch_size = 8
        attack_steps = 50

        def generate_attack(images, labels):
            return deepfool_attack(
                model=model,
                images=images,
                labels=labels,
                max_steps=50,
                overshoot=0.02,
                num_classes=10,
            )

        run_parameter = "steps50"

    elif attack_id == "cw_l2":
        batch_size = 32
        attack_steps = 100

        def generate_attack(images, labels):
            return cw_l2_attack(
                model=model,
                images=images,
                labels=labels,
                c=1.0,
                kappa=0.0,
                learning_rate=0.01,
                steps=100,
            )

        run_parameter = "steps100"

    else:
        raise ValueError(
            f"Unsupported attack_id: {attack_id}"
        )

    loader = DataLoader(
        Subset(TEST_DATASET, indices),
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=DEVICE.type == "cuda",
    )

    total_samples = 0
    clean_correct_total = 0
    adversarial_correct_total = 0
    successful_total = 0

    l2_sum = 0.0
    linf_sum = 0.0
    successful_l2_sum = 0.0
    successful_linf_sum = 0.0

    attack_time = 0.0
    clean_inference_time = 0.0
    adversarial_inference_time = 0.0

    saved_examples = {
        "originals": [],
        "adversarials": [],
        "labels": [],
        "clean_logits": [],
        "adversarial_logits": [],
    }

    synchronize_device()
    complete_start = time.perf_counter()

    for images, labels in loader:
        images = images.to(
            DEVICE,
            non_blocking=True,
        )
        labels = labels.to(
            DEVICE,
            non_blocking=True,
        )

        synchronize_device()
        clean_start = time.perf_counter()

        with torch.no_grad():
            clean_logits = model(images)

        synchronize_device()
        clean_inference_time += (
            time.perf_counter() - clean_start
        )

        synchronize_device()
        attack_start = time.perf_counter()

        adversarial_images = generate_attack(
            images,
            labels,
        )

        synchronize_device()
        attack_time += (
            time.perf_counter() - attack_start
        )

        assert adversarial_images.shape == images.shape
        assert adversarial_images.dtype == images.dtype
        assert adversarial_images.device == images.device
        assert adversarial_images.min().item() >= 0.0
        assert adversarial_images.max().item() <= 1.0

        synchronize_device()
        adversarial_start = time.perf_counter()

        with torch.no_grad():
            adversarial_logits = model(
                adversarial_images
            )

        synchronize_device()
        adversarial_inference_time += (
            time.perf_counter()
            - adversarial_start
        )

        clean_predictions = clean_logits.argmax(dim=1)
        adversarial_predictions = (
            adversarial_logits.argmax(dim=1)
        )

        clean_correct = clean_predictions.eq(labels)
        adversarial_correct = (
            adversarial_predictions.eq(labels)
        )

        successful = (
            clean_correct
            & (~adversarial_correct)
        )

        perturbation = (
            adversarial_images - images
        ).flatten(1)

        l2_norms = perturbation.norm(
            p=2,
            dim=1,
        )

        linf_norms = perturbation.abs().amax(
            dim=1,
        )

        batch_samples = images.shape[0]

        total_samples += batch_samples
        clean_correct_total += (
            clean_correct.sum().item()
        )
        adversarial_correct_total += (
            adversarial_correct.sum().item()
        )
        successful_total += (
            successful.sum().item()
        )

        l2_sum += l2_norms.sum().item()
        linf_sum += linf_norms.sum().item()

        if bool(successful.any()):
            successful_l2_sum += (
                l2_norms[successful].sum().item()
            )
            successful_linf_sum += (
                linf_norms[successful].sum().item()
            )

            already_saved = sum(
                tensor.shape[0]
                for tensor in saved_examples["labels"]
            )

            remaining = max(8 - already_saved, 0)

            if remaining > 0:
                selected = torch.where(
                    successful
                )[0][:remaining]

                saved_examples["originals"].append(
                    images[selected].detach().cpu()
                )
                saved_examples["adversarials"].append(
                    adversarial_images[
                        selected
                    ].detach().cpu()
                )
                saved_examples["labels"].append(
                    labels[selected].detach().cpu()
                )
                saved_examples["clean_logits"].append(
                    clean_logits[
                        selected
                    ].detach().cpu()
                )
                saved_examples[
                    "adversarial_logits"
                ].append(
                    adversarial_logits[
                        selected
                    ].detach().cpu()
                )

    synchronize_device()
    total_time = (
        time.perf_counter() - complete_start
    )

    attack_success_rate = (
        successful_total / clean_correct_total
        if clean_correct_total > 0
        else float("nan")
    )

    mean_l2_successful = (
        successful_l2_sum / successful_total
        if successful_total > 0
        else float("nan")
    )

    mean_linf_successful = (
        successful_linf_sum / successful_total
        if successful_total > 0
        else float("nan")
    )

    row = {
        "run_id": (
            f"{defense_id}__{attack_id}__"
            f"{run_parameter}__seed{SEED}"
        ),
        "model_id": "resnet20",
        "defense_id": defense_id,
        "attack_id": attack_id,
        "split": "test",
        "num_samples": total_samples,
        "seed": SEED,
        "epsilon": np.nan,
        "alpha": np.nan,
        "attack_steps": attack_steps,
        "clean_accuracy": (
            clean_correct_total / total_samples
        ),
        "robust_accuracy": (
            adversarial_correct_total / total_samples
        ),
        "attack_success_rate": (
            attack_success_rate
        ),
        "mean_l2": l2_sum / total_samples,
        "mean_l2_successful": (
            mean_l2_successful
        ),
        "mean_linf": (
            linf_sum / total_samples
        ),
        "mean_linf_successful": (
            mean_linf_successful
        ),
        "attack_time_seconds": attack_time,
        "defense_time_seconds": 0.0,
        "total_time_seconds": total_time,
        "checkpoint_name": (
            CHECKPOINT_NAMES[defense_id]
        ),
        "clean_inference_time_seconds": (
            clean_inference_time
        ),
        "adversarial_inference_time_seconds": (
            adversarial_inference_time
        ),
        "successful_samples": successful_total,
        "clean_correct_samples": (
            clean_correct_total
        ),
    }

    concatenated_examples = {}

    for key, tensors in saved_examples.items():
        if tensors:
            concatenated_examples[key] = torch.cat(
                tensors,
                dim=0,
            )
        else:
            concatenated_examples[key] = None

    return row, concatenated_examples


print("EXPENSIVE ATTACK EVALUATOR DEFINED.")

EXPENSIVE ATTACK EVALUATOR DEFINED.


In [10]:
calibration_rows = []
calibration_examples = {}

for defense_id in ("none", "pgd_at"):
    for attack_id in ("deepfool", "cw_l2"):
        print("=" * 70)
        print(
            "Starting:",
            defense_id,
            "vs",
            attack_id,
        )

        row, examples = evaluate_expensive_attack(
            defense_id=defense_id,
            attack_id=attack_id,
            indices=CALIBRATION_INDICES,
        )

        calibration_rows.append(row)
        calibration_examples[
            (defense_id, attack_id)
        ] = examples

        print(
            f"Clean accuracy: "
            f"{100 * row['clean_accuracy']:.2f}%"
        )
        print(
            f"Robust accuracy: "
            f"{100 * row['robust_accuracy']:.2f}%"
        )
        print(
            f"Attack success rate: "
            f"{100 * row['attack_success_rate']:.2f}%"
        )
        print(
            f"Mean L2: "
            f"{row['mean_l2']:.6f}"
        )
        print(
            f"Successful mean L2: "
            f"{row['mean_l2_successful']:.6f}"
        )
        print(
            f"Attack time: "
            f"{row['attack_time_seconds']:.2f}s"
        )

        gc.collect()

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()


calibration_df = pd.DataFrame(
    calibration_rows
)

display_columns = [
    "defense_id",
    "attack_id",
    "num_samples",
    "clean_accuracy",
    "robust_accuracy",
    "attack_success_rate",
    "mean_l2",
    "mean_l2_successful",
    "attack_time_seconds",
]

display(calibration_df[display_columns])


# --------------------------------------------------
# ذخیره دائمی نتایج کالیبراسیون در Drive
# --------------------------------------------------

calibration_csv_path = (
    DRIVE_RESULTS_DIR
    / "evaluate_colab_attack_calibration.csv"
)

calibration_indices_path = (
    DRIVE_RESULTS_DIR
    / "attack_calibration_indices.json"
)

calibration_examples_path = (
    DRIVE_RESULTS_DIR
    / "attack_calibration_examples.pt"
)

calibration_df.to_csv(
    calibration_csv_path,
    index=False,
)

calibration_indices_path.write_text(
    json.dumps(
        {
            "seed": SEED,
            "num_samples": len(
                CALIBRATION_INDICES
            ),
            "indices": CALIBRATION_INDICES,
        },
        indent=2,
    ),
    encoding="utf-8",
)

torch.save(
    calibration_examples,
    calibration_examples_path,
)

print("\nSaved calibration artifacts:")
print("-", calibration_csv_path)
print("-", calibration_indices_path)
print("-", calibration_examples_path)

print(
    "\nPHASE-2A DEEPFOOL/CW "
    "CALIBRATION COMPLETED."
)

Starting: none vs deepfool
Clean accuracy: 89.06%
Robust accuracy: 6.25%
Attack success rate: 100.00%
Mean L2: 0.101025
Successful mean L2: 0.108804
Attack time: 4.90s
Starting: none vs cw_l2
Clean accuracy: 89.06%
Robust accuracy: 0.00%
Attack success rate: 100.00%
Mean L2: 0.151434
Successful mean L2: 0.170031
Attack time: 1.64s
Starting: pgd_at vs deepfool
Clean accuracy: 67.19%
Robust accuracy: 17.19%
Attack success rate: 100.00%
Mean L2: 0.803800
Successful mean L2: 1.011080
Attack time: 5.29s
Starting: pgd_at vs cw_l2
Clean accuracy: 67.19%
Robust accuracy: 25.00%
Attack success rate: 62.79%
Mean L2: 0.347788
Successful mean L2: 0.459665
Attack time: 1.57s


,defense_id,attack_id,num_samples,clean_accuracy,robust_accuracy,attack_success_rate,mean_l2,mean_l2_successful,attack_time_seconds
0,none,deepfool,64,0.890625,0.062500,1.000000,0.101025,0.108804,4.900472
1,none,cw_l2,64,0.890625,0.000000,1.000000,0.151434,0.170031,1.640810
2,pgd_at,deepfool,64,0.671875,0.171875,1.000000,0.803800,1.011080,5.288857
3,pgd_at,cw_l2,64,0.671875,0.250000,0.627907,0.347788,0.459665,1.566200



Saved calibration artifacts:
- /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/evaluate_colab_attack_calibration.csv
- /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/attack_calibration_indices.json
- /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/attack_calibration_examples.pt

PHASE-2A DEEPFOOL/CW CALIBRATION COMPLETED.


In [11]:
import hashlib


CANONICAL_METRIC_COLUMNS = [
    "run_id",
    "model_id",
    "defense_id",
    "attack_id",
    "split",
    "num_samples",
    "seed",
    "epsilon",
    "alpha",
    "attack_steps",
    "clean_accuracy",
    "robust_accuracy",
    "attack_success_rate",
    "mean_l2",
    "mean_l2_successful",
    "mean_linf",
    "mean_linf_successful",
    "attack_time_seconds",
    "defense_time_seconds",
    "total_time_seconds",
    "checkpoint_name",
]


# این بار کل CIFAR-10 test set استفاده می‌شود.
FINAL_EXPENSIVE_INDICES = list(
    range(len(TEST_DATASET))
)

FINAL_EXPENSIVE_SAMPLES = len(
    FINAL_EXPENSIVE_INDICES
)

FINAL_STANDARD_METRICS_PATH = (
    DRIVE_RESULTS_DIR
    / "evaluate_colab_expensive_standard_final.csv"
)

FINAL_EXAMPLES_DIRECTORY = (
    DRIVE_RESULTS_DIR
    / "final_attack_examples"
)

FINAL_EXAMPLES_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


if FINAL_STANDARD_METRICS_PATH.is_file():
    final_standard_df = pd.read_csv(
        FINAL_STANDARD_METRICS_PATH
    )

    print(
        "Existing partial results loaded:",
        len(final_standard_df),
        "rows",
    )

else:
    final_standard_df = pd.DataFrame(
        columns=CANONICAL_METRIC_COLUMNS
    )

    print("No previous final results found.")


def final_run_is_complete(
    metrics_df,
    defense_id,
    attack_id,
):
    if metrics_df.empty:
        return False

    required_columns = {
        "defense_id",
        "attack_id",
        "num_samples",
        "robust_accuracy",
        "attack_success_rate",
    }

    if not required_columns.issubset(
        metrics_df.columns
    ):
        return False

    selected = metrics_df.loc[
        metrics_df["defense_id"].eq(defense_id)
        & metrics_df["attack_id"].eq(attack_id)
        & metrics_df["num_samples"].eq(
            FINAL_EXPENSIVE_SAMPLES
        )
    ]

    if len(selected) != 1:
        return False

    row = selected.iloc[0]

    return (
        pd.notna(row["robust_accuracy"])
        and pd.notna(row["attack_success_rate"])
    )


print("Final samples:", FINAL_EXPENSIVE_SAMPLES)
print("Metrics path:", FINAL_STANDARD_METRICS_PATH)
print("Examples directory:", FINAL_EXAMPLES_DIRECTORY)

print("\nFINAL EXPENSIVE EVALUATION SETUP PASSED.")

No previous final results found.
Final samples: 10000
Metrics path: /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/evaluate_colab_expensive_standard_final.csv
Examples directory: /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/final_attack_examples

FINAL EXPENSIVE EVALUATION SETUP PASSED.


In [12]:
FINAL_EXPENSIVE_RUNS = [
    ("none", "deepfool"),
    ("none", "cw_l2"),
    ("pgd_at", "deepfool"),
    ("pgd_at", "cw_l2"),
]


for defense_id, attack_id in FINAL_EXPENSIVE_RUNS:
    if final_run_is_complete(
        final_standard_df,
        defense_id,
        attack_id,
    ):
        print("=" * 70)
        print(
            "SKIPPED — already completed:",
            defense_id,
            "vs",
            attack_id,
        )
        continue

    print("=" * 70)
    print(
        "FINAL RUN STARTED:",
        defense_id,
        "vs",
        attack_id,
    )
    print(
        "Samples:",
        FINAL_EXPENSIVE_SAMPLES,
    )

    row, examples = evaluate_expensive_attack(
        defense_id=defense_id,
        attack_id=attack_id,
        indices=FINAL_EXPENSIVE_INDICES,
    )

    # اگر نسخه ناقصی از همین آزمایش وجود داشته باشد،
    # قبل از افزودن نتیجه نهایی حذف می‌شود.
    if not final_standard_df.empty:
        duplicate_mask = (
            final_standard_df[
                "defense_id"
            ].eq(defense_id)
            & final_standard_df[
                "attack_id"
            ].eq(attack_id)
        )

        final_standard_df = (
            final_standard_df.loc[
                ~duplicate_mask
            ].copy()
        )

    canonical_row = {
        column: row[column]
        for column in CANONICAL_METRIC_COLUMNS
    }

    final_standard_df = pd.concat(
        [
            final_standard_df,
            pd.DataFrame([canonical_row]),
        ],
        ignore_index=True,
    )

    final_standard_df = (
        final_standard_df[
            CANONICAL_METRIC_COLUMNS
        ]
        .sort_values(
            ["defense_id", "attack_id"]
        )
        .reset_index(drop=True)
    )

    # ابتدا در فایل موقت نوشته می‌شود و بعد جایگزین می‌گردد.
    temporary_metrics_path = (
        FINAL_STANDARD_METRICS_PATH
        .with_suffix(".tmp")
    )

    final_standard_df.to_csv(
        temporary_metrics_path,
        index=False,
    )

    temporary_metrics_path.replace(
        FINAL_STANDARD_METRICS_PATH
    )

    examples_path = (
        FINAL_EXAMPLES_DIRECTORY
        / (
            f"{defense_id}__"
            f"{attack_id}__examples.pt"
        )
    )

    torch.save(
        examples,
        examples_path,
    )

    metrics_hash = hashlib.sha256(
        FINAL_STANDARD_METRICS_PATH.read_bytes()
    ).hexdigest()

    print(
        f"Clean accuracy: "
        f"{100 * row['clean_accuracy']:.2f}%"
    )
    print(
        f"Robust accuracy: "
        f"{100 * row['robust_accuracy']:.2f}%"
    )
    print(
        f"Attack success rate: "
        f"{100 * row['attack_success_rate']:.2f}%"
    )
    print(
        f"Mean L2 successful: "
        f"{row['mean_l2_successful']:.6f}"
    )
    print(
        f"Attack time: "
        f"{row['attack_time_seconds']:.2f}s"
    )
    print("Saved metrics:", FINAL_STANDARD_METRICS_PATH)
    print("Saved examples:", examples_path)
    print("Metrics SHA256:", metrics_hash)

    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


print("\nCURRENT FINAL RESULTS:")

display(
    final_standard_df[
        [
            "defense_id",
            "attack_id",
            "num_samples",
            "clean_accuracy",
            "robust_accuracy",
            "attack_success_rate",
            "mean_l2_successful",
            "attack_time_seconds",
        ]
    ]
)

FINAL RUN STARTED: none vs deepfool
Samples: 10000


/tmp/ipykernel_793/2226025370.py:65: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_standard_df = pd.concat(


Clean accuracy: 84.03%
Robust accuracy: 9.82%
Attack success rate: 100.00%
Mean L2 successful: 0.117538
Attack time: 823.55s
Saved metrics: /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/evaluate_colab_expensive_standard_final.csv
Saved examples: /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/final_attack_examples/none__deepfool__examples.pt
Metrics SHA256: 88a6540ab975248fc799a235a66fc043978d1263532a3e25c04cc78e3678033d
FINAL RUN STARTED: none vs cw_l2
Samples: 10000
Clean accuracy: 84.03%
Robust accuracy: 0.00%
Attack success rate: 100.00%
Mean L2 successful: 0.186184
Attack time: 247.56s
Saved metrics: /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/evaluate_colab_expensive_standard_final.csv
Saved examples: /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/final_attack_examples/none__cw_l2__examples.pt
Metrics SHA256: 83f381f398f49f297984afe1e5f8eb3185f11c1d08ed1608c8e10ed4635a776b
FINAL RUN STARTED

,defense_id,attack_id,num_samples,clean_accuracy,robust_accuracy,attack_success_rate,mean_l2_successful,attack_time_seconds
0,none,cw_l2,10000,0.8403,0.0000,1.000000,0.186184,247.555366
1,none,deepfool,10000,0.8403,0.0982,1.000000,0.117538,823.553014
2,pgd_at,cw_l2,10000,0.6798,0.3565,0.475581,0.523592,249.479599
3,pgd_at,deepfool,10000,0.6798,0.1473,0.999853,1.262114,774.959111


In [13]:
assert FINAL_STANDARD_METRICS_PATH.is_file()

verified_final_df = pd.read_csv(
    FINAL_STANDARD_METRICS_PATH
)

assert list(
    verified_final_df.columns
) == CANONICAL_METRIC_COLUMNS

assert len(verified_final_df) == 4

assert set(
    verified_final_df["defense_id"]
) == {"none", "pgd_at"}

assert set(
    verified_final_df["attack_id"]
) == {"deepfool", "cw_l2"}

assert verified_final_df[
    "num_samples"
].eq(10_000).all()

for accuracy_column in (
    "clean_accuracy",
    "robust_accuracy",
    "attack_success_rate",
):
    assert verified_final_df[
        accuracy_column
    ].between(0.0, 1.0).all()

assert verified_final_df[
    "epsilon"
].isna().all()

assert verified_final_df[
    "alpha"
].isna().all()


for defense_id, attack_id in FINAL_EXPENSIVE_RUNS:
    examples_path = (
        FINAL_EXAMPLES_DIRECTORY
        / (
            f"{defense_id}__"
            f"{attack_id}__examples.pt"
        )
    )

    assert examples_path.is_file()

    examples = torch.load(
        examples_path,
        map_location="cpu",
        weights_only=False,
    )

    assert examples["originals"] is not None
    assert examples["adversarials"] is not None
    assert examples["labels"] is not None

    assert examples["originals"].shape[0] >= 2
    assert (
        examples["originals"].shape
        == examples["adversarials"].shape
    )

    print(
        "VERIFIED:",
        examples_path.name,
        "successful examples:",
        examples["labels"].shape[0],
    )


final_metrics_hash = hashlib.sha256(
    FINAL_STANDARD_METRICS_PATH.read_bytes()
).hexdigest()

print("\nFinal metrics SHA256:", final_metrics_hash)
print(
    "\nALL FINAL DEEPFOOL/CW "
    "STANDARD EVALUATIONS PASSED."
)

VERIFIED: none__deepfool__examples.pt successful examples: 8
VERIFIED: none__cw_l2__examples.pt successful examples: 8
VERIFIED: pgd_at__deepfool__examples.pt successful examples: 8
VERIFIED: pgd_at__cw_l2__examples.pt successful examples: 8

Final metrics SHA256: 68a164455dc0e271efeaa3ef58a158ccd9bd4eff5e03bd01fb1ed89ddd46ab36

ALL FINAL DEEPFOOL/CW STANDARD EVALUATIONS PASSED.


In [14]:
from torch.utils.data import DataLoader, Subset

import gc
import hashlib
import time

import numpy as np
import pandas as pd
import torch

from src.defenses.randomized_smoothing import (
    predict_smoothed,
)


SMOOTH_SIGMA = 0.25
SMOOTH_NUM_SAMPLES = 100
SMOOTH_NOISE_BATCH_SIZE = 25
SMOOTH_NUM_CLASSES = 10

SMOOTHING_MODEL = CLASSIFIERS["rand_smooth"]
SMOOTHING_MODEL.eval()

RS_CALIBRATION_INDICES = list(
    CALIBRATION_INDICES
)


def build_smoothed_clean_cache(
    indices,
    batch_size=16,
):
    loader = DataLoader(
        Subset(TEST_DATASET, indices),
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=DEVICE.type == "cuda",
    )

    predictions_list = []
    vote_counts_list = []
    labels_list = []

    generator = torch.Generator(
        device=DEVICE,
    ).manual_seed(SEED)

    synchronize_device()
    defense_start = time.perf_counter()

    for images, labels in loader:
        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        predictions, vote_counts = predict_smoothed(
            model=SMOOTHING_MODEL,
            images=images,
            sigma=SMOOTH_SIGMA,
            num_samples=SMOOTH_NUM_SAMPLES,
            noise_batch_size=SMOOTH_NOISE_BATCH_SIZE,
            num_classes=SMOOTH_NUM_CLASSES,
            clip_noise=True,
            generator=generator,
        )

        assert predictions.shape == (
            images.shape[0],
        )

        assert vote_counts.shape == (
            images.shape[0],
            SMOOTH_NUM_CLASSES,
        )

        assert torch.all(
            vote_counts.sum(dim=1)
            == SMOOTH_NUM_SAMPLES
        )

        predictions_list.append(
            predictions.detach().cpu()
        )
        vote_counts_list.append(
            vote_counts.detach().cpu()
        )
        labels_list.append(labels.cpu())

    synchronize_device()
    defense_time = (
        time.perf_counter() - defense_start
    )

    predictions = torch.cat(
        predictions_list,
        dim=0,
    )
    vote_counts = torch.cat(
        vote_counts_list,
        dim=0,
    )
    labels = torch.cat(
        labels_list,
        dim=0,
    )

    clean_accuracy = (
        predictions.eq(labels)
        .float()
        .mean()
        .item()
    )

    mean_voting_confidence = (
        vote_counts.max(dim=1).values
        .float()
        .div(SMOOTH_NUM_SAMPLES)
        .mean()
        .item()
    )

    return {
        "indices": list(indices),
        "predictions": predictions,
        "vote_counts": vote_counts,
        "labels": labels,
        "clean_accuracy": clean_accuracy,
        "mean_voting_confidence": (
            mean_voting_confidence
        ),
        "defense_time_seconds": defense_time,
    }


rs_clean_cache = build_smoothed_clean_cache(
    RS_CALIBRATION_INDICES,
)

RS_CLEAN_CACHE_PATH = (
    DRIVE_RESULTS_DIR
    / "rand_smooth_calibration_clean_cache.pt"
)

torch.save(
    rs_clean_cache,
    RS_CLEAN_CACHE_PATH,
)

print(
    "Samples:",
    len(RS_CALIBRATION_INDICES),
)
print(
    "Smoothed clean accuracy:",
    f"{100 * rs_clean_cache['clean_accuracy']:.2f}%",
)
print(
    "Mean voting confidence:",
    f"{100 * rs_clean_cache['mean_voting_confidence']:.2f}%",
)
print(
    "Clean defense time:",
    f"{rs_clean_cache['defense_time_seconds']:.2f}s",
)
print("Saved:", RS_CLEAN_CACHE_PATH)

print(
    "\nRANDOMIZED-SMOOTHING CLEAN "
    "CALIBRATION PASSED."
)

Samples: 64
Smoothed clean accuracy: 76.56%
Mean voting confidence: 79.09%
Clean defense time: 0.47s
Saved: /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/rand_smooth_calibration_clean_cache.pt

RANDOMIZED-SMOOTHING CLEAN CALIBRATION PASSED.


In [15]:
def evaluate_rand_smooth_attack(
    attack_id,
    indices,
    clean_cache,
):
    model = SMOOTHING_MODEL
    model.eval()

    if attack_id == "deepfool":
        batch_size = 8
        attack_steps = 50
        run_parameter = "steps50"

        def generate_attack(images, labels):
            return deepfool_attack(
                model=model,
                images=images,
                labels=labels,
                max_steps=50,
                overshoot=0.02,
                num_classes=10,
            )

    elif attack_id == "cw_l2":
        batch_size = 32
        attack_steps = 100
        run_parameter = "steps100"

        def generate_attack(images, labels):
            return cw_l2_attack(
                model=model,
                images=images,
                labels=labels,
                c=1.0,
                kappa=0.0,
                learning_rate=0.01,
                steps=100,
            )

    else:
        raise ValueError(
            f"Unsupported attack: {attack_id}"
        )

    assert clean_cache["indices"] == list(indices)
    assert len(clean_cache["predictions"]) == len(
        indices
    )

    loader = DataLoader(
        Subset(TEST_DATASET, indices),
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=DEVICE.type == "cuda",
    )

    total_samples = 0
    clean_correct_total = 0
    adversarial_correct_total = 0
    successful_total = 0

    l2_sum = 0.0
    linf_sum = 0.0
    successful_l2_sum = 0.0
    successful_linf_sum = 0.0

    attack_time = 0.0
    adversarial_defense_time = 0.0

    saved_examples = {
        "originals": [],
        "adversarials": [],
        "labels": [],
        "clean_logits": [],
        "adversarial_logits": [],
        "clean_predictions": [],
        "adversarial_predictions": [],
        "clean_vote_counts": [],
        "adversarial_vote_counts": [],
    }

    adversarial_generator = torch.Generator(
        device=DEVICE,
    ).manual_seed(SEED)

    cache_offset = 0

    synchronize_device()
    evaluation_start = time.perf_counter()

    for images, labels in loader:
        images = images.to(
            DEVICE,
            non_blocking=True,
        )
        labels = labels.to(
            DEVICE,
            non_blocking=True,
        )

        batch_samples = images.shape[0]
        cache_slice = slice(
            cache_offset,
            cache_offset + batch_samples,
        )

        cached_labels = clean_cache[
            "labels"
        ][cache_slice].to(DEVICE)

        assert torch.equal(
            labels,
            cached_labels,
        )

        clean_predictions = clean_cache[
            "predictions"
        ][cache_slice].to(DEVICE)

        clean_vote_counts = clean_cache[
            "vote_counts"
        ][cache_slice].to(DEVICE)

        synchronize_device()
        attack_start = time.perf_counter()

        adversarial_images = generate_attack(
            images,
            labels,
        )

        synchronize_device()
        attack_time += (
            time.perf_counter() - attack_start
        )

        assert adversarial_images.shape == images.shape
        assert adversarial_images.dtype == images.dtype
        assert adversarial_images.device == images.device
        assert adversarial_images.min().item() >= 0.0
        assert adversarial_images.max().item() <= 1.0

        synchronize_device()
        defense_start = time.perf_counter()

        (
            adversarial_predictions,
            adversarial_vote_counts,
        ) = predict_smoothed(
            model=model,
            images=adversarial_images,
            sigma=SMOOTH_SIGMA,
            num_samples=SMOOTH_NUM_SAMPLES,
            noise_batch_size=SMOOTH_NOISE_BATCH_SIZE,
            num_classes=SMOOTH_NUM_CLASSES,
            clip_noise=True,
            generator=adversarial_generator,
        )

        synchronize_device()
        adversarial_defense_time += (
            time.perf_counter() - defense_start
        )

        assert torch.all(
            adversarial_vote_counts.sum(dim=1)
            == SMOOTH_NUM_SAMPLES
        )

        clean_correct = clean_predictions.eq(
            labels
        )

        adversarial_correct = (
            adversarial_predictions.eq(labels)
        )

        successful = (
            clean_correct
            & (~adversarial_correct)
        )

        perturbation = (
            adversarial_images - images
        ).flatten(1)

        l2_norms = perturbation.norm(
            p=2,
            dim=1,
        )

        linf_norms = perturbation.abs().amax(
            dim=1,
        )

        total_samples += batch_samples
        clean_correct_total += (
            clean_correct.sum().item()
        )
        adversarial_correct_total += (
            adversarial_correct.sum().item()
        )
        successful_total += (
            successful.sum().item()
        )

        l2_sum += l2_norms.sum().item()
        linf_sum += linf_norms.sum().item()

        if bool(successful.any()):
            successful_l2_sum += (
                l2_norms[successful].sum().item()
            )
            successful_linf_sum += (
                linf_norms[successful].sum().item()
            )

            already_saved = sum(
                tensor.shape[0]
                for tensor in saved_examples["labels"]
            )

            remaining = max(
                8 - already_saved,
                0,
            )

            if remaining > 0:
                selected = torch.where(
                    successful
                )[0][:remaining]

                with torch.no_grad():
                    clean_logits = model(
                        images[selected]
                    )
                    adversarial_logits = model(
                        adversarial_images[selected]
                    )

                saved_examples["originals"].append(
                    images[selected].detach().cpu()
                )
                saved_examples["adversarials"].append(
                    adversarial_images[
                        selected
                    ].detach().cpu()
                )
                saved_examples["labels"].append(
                    labels[selected].detach().cpu()
                )
                saved_examples["clean_logits"].append(
                    clean_logits.detach().cpu()
                )
                saved_examples[
                    "adversarial_logits"
                ].append(
                    adversarial_logits.detach().cpu()
                )
                saved_examples[
                    "clean_predictions"
                ].append(
                    clean_predictions[
                        selected
                    ].detach().cpu()
                )
                saved_examples[
                    "adversarial_predictions"
                ].append(
                    adversarial_predictions[
                        selected
                    ].detach().cpu()
                )
                saved_examples[
                    "clean_vote_counts"
                ].append(
                    clean_vote_counts[
                        selected
                    ].detach().cpu()
                )
                saved_examples[
                    "adversarial_vote_counts"
                ].append(
                    adversarial_vote_counts[
                        selected
                    ].detach().cpu()
                )

        cache_offset += batch_samples

    synchronize_device()
    attack_evaluation_time = (
        time.perf_counter() - evaluation_start
    )

    assert cache_offset == len(indices)

    clean_defense_time = clean_cache[
        "defense_time_seconds"
    ]

    defense_time = (
        clean_defense_time
        + adversarial_defense_time
    )

    total_time = (
        clean_defense_time
        + attack_evaluation_time
    )

    attack_success_rate = (
        successful_total / clean_correct_total
        if clean_correct_total > 0
        else float("nan")
    )

    mean_l2_successful = (
        successful_l2_sum / successful_total
        if successful_total > 0
        else float("nan")
    )

    mean_linf_successful = (
        successful_linf_sum / successful_total
        if successful_total > 0
        else float("nan")
    )

    row = {
        "run_id": (
            f"rand_smooth__{attack_id}__"
            f"{run_parameter}__sigma0p25__"
            f"votes100__seed{SEED}"
        ),
        "model_id": "resnet20",
        "defense_id": "rand_smooth",
        "attack_id": attack_id,
        "split": "test",
        "num_samples": total_samples,
        "seed": SEED,
        "epsilon": np.nan,
        "alpha": np.nan,
        "attack_steps": attack_steps,
        "clean_accuracy": (
            clean_correct_total / total_samples
        ),
        "robust_accuracy": (
            adversarial_correct_total
            / total_samples
        ),
        "attack_success_rate": (
            attack_success_rate
        ),
        "mean_l2": l2_sum / total_samples,
        "mean_l2_successful": (
            mean_l2_successful
        ),
        "mean_linf": (
            linf_sum / total_samples
        ),
        "mean_linf_successful": (
            mean_linf_successful
        ),
        "attack_time_seconds": attack_time,
        "defense_time_seconds": defense_time,
        "total_time_seconds": total_time,
        "checkpoint_name": (
            CHECKPOINT_NAMES["rand_smooth"]
        ),
    }

    concatenated_examples = {}

    for key, tensors in saved_examples.items():
        concatenated_examples[key] = (
            torch.cat(tensors, dim=0)
            if tensors
            else None
        )

    return row, concatenated_examples


print(
    "RANDOMIZED-SMOOTHING ATTACK "
    "EVALUATOR DEFINED."
)

RANDOMIZED-SMOOTHING ATTACK EVALUATOR DEFINED.


In [16]:
RS_CALIBRATION_METRICS_PATH = (
    DRIVE_RESULTS_DIR
    / (
        "evaluate_colab_rand_smooth_"
        "expensive_calibration.csv"
    )
)

RS_CALIBRATION_EXAMPLES_PATH = (
    DRIVE_RESULTS_DIR
    / (
        "rand_smooth_expensive_"
        "calibration_examples.pt"
    )
)

rs_calibration_rows = []
rs_calibration_examples = {}


for attack_id in ("deepfool", "cw_l2"):
    print("=" * 70)
    print(
        "Starting: rand_smooth vs",
        attack_id,
    )

    row, examples = evaluate_rand_smooth_attack(
        attack_id=attack_id,
        indices=RS_CALIBRATION_INDICES,
        clean_cache=rs_clean_cache,
    )

    rs_calibration_rows.append(row)
    rs_calibration_examples[attack_id] = examples

    current_df = pd.DataFrame(
        rs_calibration_rows
    )[CANONICAL_METRIC_COLUMNS]

    temporary_path = (
        RS_CALIBRATION_METRICS_PATH
        .with_suffix(".tmp")
    )

    current_df.to_csv(
        temporary_path,
        index=False,
    )

    temporary_path.replace(
        RS_CALIBRATION_METRICS_PATH
    )

    torch.save(
        rs_calibration_examples,
        RS_CALIBRATION_EXAMPLES_PATH,
    )

    print(
        "Clean accuracy:",
        f"{100 * row['clean_accuracy']:.2f}%",
    )
    print(
        "Robust accuracy:",
        f"{100 * row['robust_accuracy']:.2f}%",
    )
    print(
        "Attack success rate:",
        f"{100 * row['attack_success_rate']:.2f}%",
    )
    print(
        "Mean L2 successful:",
        f"{row['mean_l2_successful']:.6f}",
    )
    print(
        "Attack time:",
        f"{row['attack_time_seconds']:.2f}s",
    )
    print(
        "Defense time:",
        f"{row['defense_time_seconds']:.2f}s",
    )
    print(
        "Total time:",
        f"{row['total_time_seconds']:.2f}s",
    )

    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


rs_calibration_df = pd.read_csv(
    RS_CALIBRATION_METRICS_PATH
)

assert len(rs_calibration_df) == 2
assert set(
    rs_calibration_df["attack_id"]
) == {"deepfool", "cw_l2"}

assert rs_calibration_df[
    "defense_id"
].eq("rand_smooth").all()

assert rs_calibration_df[
    "num_samples"
].eq(64).all()

assert rs_calibration_df[
    "clean_accuracy"
].nunique() == 1


for attack_id, examples in (
    rs_calibration_examples.items()
):
    labels = examples["labels"]

    if labels is None:
        successful_examples = 0
    else:
        successful_examples = labels.shape[0]

        assert torch.all(
            examples[
                "clean_vote_counts"
            ].sum(dim=1)
            == SMOOTH_NUM_SAMPLES
        )

        assert torch.all(
            examples[
                "adversarial_vote_counts"
            ].sum(dim=1)
            == SMOOTH_NUM_SAMPLES
        )

    print(
        "VERIFIED:",
        attack_id,
        "successful examples:",
        successful_examples,
    )


display(
    rs_calibration_df[
        [
            "defense_id",
            "attack_id",
            "num_samples",
            "clean_accuracy",
            "robust_accuracy",
            "attack_success_rate",
            "mean_l2_successful",
            "attack_time_seconds",
            "defense_time_seconds",
            "total_time_seconds",
        ]
    ]
)

rs_calibration_hash = hashlib.sha256(
    RS_CALIBRATION_METRICS_PATH.read_bytes()
).hexdigest()

print(
    "\nCalibration SHA256:",
    rs_calibration_hash,
)
print(
    "\nRANDOMIZED-SMOOTHING "
    "DEEPFOOL/CW CALIBRATION PASSED."
)

Starting: rand_smooth vs deepfool
Clean accuracy: 76.56%
Robust accuracy: 70.31%
Attack success rate: 10.20%
Mean L2 successful: 0.332065
Attack time: 4.50s
Defense time: 0.81s
Total time: 5.46s
Starting: rand_smooth vs cw_l2
Clean accuracy: 76.56%
Robust accuracy: 68.75%
Attack success rate: 10.20%
Mean L2 successful: 0.331423
Attack time: 1.54s
Defense time: 0.77s
Total time: 2.43s
VERIFIED: deepfool successful examples: 5
VERIFIED: cw_l2 successful examples: 5


,defense_id,attack_id,num_samples,clean_accuracy,robust_accuracy,attack_success_rate,mean_l2_successful,attack_time_seconds,defense_time_seconds,total_time_seconds
0,rand_smooth,deepfool,64,0.765625,0.703125,0.102041,0.332065,4.497183,0.810352,5.458311
1,rand_smooth,cw_l2,64,0.765625,0.687500,0.102041,0.331423,1.537565,0.772380,2.431153



Calibration SHA256: f997a560efd335607347f4fe4daa2c2fb5d407b6e51486964077e9cc6ad1c8ac

RANDOMIZED-SMOOTHING DEEPFOOL/CW CALIBRATION PASSED.


In [17]:
FINAL_RS_INDICES = list(
    range(len(TEST_DATASET))
)

FINAL_RS_SAMPLES = len(
    FINAL_RS_INDICES
)

RS_FINAL_CLEAN_CACHE_PATH = (
    DRIVE_RESULTS_DIR
    / "rand_smooth_final_clean_cache.pt"
)


def rs_clean_cache_is_valid(cache):
    required_keys = {
        "indices",
        "predictions",
        "vote_counts",
        "labels",
        "clean_accuracy",
        "mean_voting_confidence",
        "defense_time_seconds",
    }

    if not required_keys.issubset(cache):
        return False

    if cache["indices"] != FINAL_RS_INDICES:
        return False

    if cache["predictions"].shape != (
        FINAL_RS_SAMPLES,
    ):
        return False

    if cache["labels"].shape != (
        FINAL_RS_SAMPLES,
    ):
        return False

    if cache["vote_counts"].shape != (
        FINAL_RS_SAMPLES,
        SMOOTH_NUM_CLASSES,
    ):
        return False

    if not bool(
        torch.all(
            cache["vote_counts"].sum(dim=1)
            == SMOOTH_NUM_SAMPLES
        )
    ):
        return False

    calculated_accuracy = (
        cache["predictions"]
        .eq(cache["labels"])
        .float()
        .mean()
        .item()
    )

    return abs(
        calculated_accuracy
        - cache["clean_accuracy"]
    ) < 1e-10


rs_final_clean_cache = None

if RS_FINAL_CLEAN_CACHE_PATH.is_file():
    try:
        loaded_cache = torch.load(
            RS_FINAL_CLEAN_CACHE_PATH,
            map_location="cpu",
            weights_only=False,
        )

        if rs_clean_cache_is_valid(loaded_cache):
            rs_final_clean_cache = loaded_cache
            print(
                "Valid existing clean cache loaded."
            )
        else:
            print(
                "Existing cache is incomplete; "
                "rebuilding it."
            )

    except Exception as error:
        print(
            "Could not load existing cache:",
            repr(error),
        )
        print("Rebuilding clean cache.")


if rs_final_clean_cache is None:
    print(
        "Building full randomized-smoothing "
        "clean cache..."
    )
    print("Samples:", FINAL_RS_SAMPLES)

    rs_final_clean_cache = (
        build_smoothed_clean_cache(
            indices=FINAL_RS_INDICES,
            batch_size=16,
        )
    )

    temporary_cache_path = (
        RS_FINAL_CLEAN_CACHE_PATH.with_name(
            RS_FINAL_CLEAN_CACHE_PATH.name
            + ".tmp"
        )
    )

    torch.save(
        rs_final_clean_cache,
        temporary_cache_path,
    )

    temporary_cache_path.replace(
        RS_FINAL_CLEAN_CACHE_PATH
    )


assert rs_clean_cache_is_valid(
    rs_final_clean_cache
)

clean_cache_hash = hashlib.sha256(
    RS_FINAL_CLEAN_CACHE_PATH.read_bytes()
).hexdigest()

print(
    "\nSamples:",
    len(rs_final_clean_cache["labels"]),
)
print(
    "Smoothed clean accuracy:",
    (
        f"{100 * rs_final_clean_cache['clean_accuracy']:.2f}%"
    ),
)
print(
    "Mean voting confidence:",
    (
        f"{100 * rs_final_clean_cache['mean_voting_confidence']:.2f}%"
    ),
)
print(
    "Clean defense time:",
    (
        f"{rs_final_clean_cache['defense_time_seconds']:.2f}s"
    ),
)
print("Saved:", RS_FINAL_CLEAN_CACHE_PATH)
print("Cache SHA256:", clean_cache_hash)

print(
    "\nFULL RANDOMIZED-SMOOTHING "
    "CLEAN CACHE PASSED."
)

Building full randomized-smoothing clean cache...
Samples: 10000

Samples: 10000
Smoothed clean accuracy: 75.15%
Mean voting confidence: 80.46%
Clean defense time: 53.73s
Saved: /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/rand_smooth_final_clean_cache.pt
Cache SHA256: 014c517172e80506839768c57559c9b408c1046939a628f1b99e616bad9cc4f8

FULL RANDOMIZED-SMOOTHING CLEAN CACHE PASSED.


In [18]:
RS_FINAL_METRICS_PATH = (
    DRIVE_RESULTS_DIR
    / "evaluate_colab_expensive_rand_smooth_final.csv"
)

RS_FINAL_EXAMPLES_DIRECTORY = (
    DRIVE_RESULTS_DIR
    / "final_attack_examples"
)

RS_FINAL_EXAMPLES_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


if RS_FINAL_METRICS_PATH.is_file():
    rs_final_df = pd.read_csv(
        RS_FINAL_METRICS_PATH
    )

    assert list(
        rs_final_df.columns
    ) == CANONICAL_METRIC_COLUMNS

    print(
        "Existing randomized-smoothing "
        "results loaded:",
        len(rs_final_df),
        "rows",
    )

else:
    rs_final_df = pd.DataFrame(
        columns=CANONICAL_METRIC_COLUMNS
    )

    print(
        "No previous full randomized-"
        "smoothing results found."
    )


def rs_final_run_is_complete(
    metrics_df,
    attack_id,
):
    if metrics_df.empty:
        return False

    selected = metrics_df.loc[
        metrics_df["defense_id"].eq(
            "rand_smooth"
        )
        & metrics_df["attack_id"].eq(
            attack_id
        )
        & metrics_df["num_samples"].eq(
            FINAL_RS_SAMPLES
        )
    ]

    if len(selected) != 1:
        return False

    row = selected.iloc[0]

    examples_path = (
        RS_FINAL_EXAMPLES_DIRECTORY
        / (
            f"rand_smooth__"
            f"{attack_id}__examples.pt"
        )
    )

    return (
        pd.notna(row["clean_accuracy"])
        and pd.notna(row["robust_accuracy"])
        and pd.notna(
            row["attack_success_rate"]
        )
        and examples_path.is_file()
    )


FINAL_RS_RUNS = (
    "deepfool",
    "cw_l2",
)


for attack_id in FINAL_RS_RUNS:
    if rs_final_run_is_complete(
        rs_final_df,
        attack_id,
    ):
        print("=" * 70)
        print(
            "SKIPPED — already completed:",
            "rand_smooth vs",
            attack_id,
        )
        continue

    print("=" * 70)
    print(
        "FINAL RUN STARTED:",
        "rand_smooth vs",
        attack_id,
    )
    print("Samples:", FINAL_RS_SAMPLES)

    row, examples = evaluate_rand_smooth_attack(
        attack_id=attack_id,
        indices=FINAL_RS_INDICES,
        clean_cache=rs_final_clean_cache,
    )

    examples_path = (
        RS_FINAL_EXAMPLES_DIRECTORY
        / (
            f"rand_smooth__"
            f"{attack_id}__examples.pt"
        )
    )

    temporary_examples_path = (
        examples_path.with_name(
            examples_path.name + ".tmp"
        )
    )

    torch.save(
        examples,
        temporary_examples_path,
    )

    temporary_examples_path.replace(
        examples_path
    )

    if not rs_final_df.empty:
        duplicate_mask = (
            rs_final_df["defense_id"].eq(
                "rand_smooth"
            )
            & rs_final_df["attack_id"].eq(
                attack_id
            )
        )

        rs_final_df = (
            rs_final_df.loc[
                ~duplicate_mask
            ].copy()
        )

    canonical_row = {
        column: row[column]
        for column in CANONICAL_METRIC_COLUMNS
    }

    new_row_df = pd.DataFrame(
        [canonical_row],
        columns=CANONICAL_METRIC_COLUMNS,
    )

    if rs_final_df.empty:
        rs_final_df = new_row_df
    else:
        rs_final_df = pd.concat(
            [
                rs_final_df,
                new_row_df,
            ],
            ignore_index=True,
        )

    rs_final_df = (
        rs_final_df[
            CANONICAL_METRIC_COLUMNS
        ]
        .sort_values(
            ["defense_id", "attack_id"]
        )
        .reset_index(drop=True)
    )

    temporary_metrics_path = (
        RS_FINAL_METRICS_PATH.with_name(
            RS_FINAL_METRICS_PATH.name
            + ".tmp"
        )
    )

    rs_final_df.to_csv(
        temporary_metrics_path,
        index=False,
    )

    temporary_metrics_path.replace(
        RS_FINAL_METRICS_PATH
    )

    metrics_hash = hashlib.sha256(
        RS_FINAL_METRICS_PATH.read_bytes()
    ).hexdigest()

    successful_examples = (
        0
        if examples["labels"] is None
        else examples["labels"].shape[0]
    )

    print(
        f"Clean accuracy: "
        f"{100 * row['clean_accuracy']:.2f}%"
    )
    print(
        f"Robust accuracy: "
        f"{100 * row['robust_accuracy']:.2f}%"
    )
    print(
        f"Attack success rate: "
        f"{100 * row['attack_success_rate']:.2f}%"
    )
    print(
        f"Mean L2 successful: "
        f"{row['mean_l2_successful']:.6f}"
    )
    print(
        f"Attack time: "
        f"{row['attack_time_seconds']:.2f}s"
    )
    print(
        f"Defense time: "
        f"{row['defense_time_seconds']:.2f}s"
    )
    print(
        f"Total time: "
        f"{row['total_time_seconds']:.2f}s"
    )
    print(
        "Successful examples saved:",
        successful_examples,
    )
    print("Saved metrics:", RS_FINAL_METRICS_PATH)
    print("Saved examples:", examples_path)
    print("Metrics SHA256:", metrics_hash)

    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


print("\nCURRENT RAND-SMOOTH FINAL RESULTS:")

display(
    rs_final_df[
        [
            "defense_id",
            "attack_id",
            "num_samples",
            "clean_accuracy",
            "robust_accuracy",
            "attack_success_rate",
            "mean_l2_successful",
            "attack_time_seconds",
            "defense_time_seconds",
            "total_time_seconds",
        ]
    ]
)

No previous full randomized-smoothing results found.
FINAL RUN STARTED: rand_smooth vs deepfool
Samples: 10000
Clean accuracy: 75.15%
Robust accuracy: 71.49%
Attack success rate: 7.54%
Mean L2 successful: 0.460774
Attack time: 710.57s
Defense time: 102.38s
Total time: 815.16s
Successful examples saved: 8
Saved metrics: /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/evaluate_colab_expensive_rand_smooth_final.csv
Saved examples: /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/final_attack_examples/rand_smooth__deepfool__examples.pt
Metrics SHA256: 1276bddd016e3a676a746e64638c1d4d6ff2a655e167f6e736ba40746269e87f
FINAL RUN STARTED: rand_smooth vs cw_l2
Samples: 10000
Clean accuracy: 75.15%
Robust accuracy: 70.95%
Attack success rate: 6.83%
Mean L2 successful: 0.317013
Attack time: 251.91s
Defense time: 102.69s
Total time: 355.33s
Successful examples saved: 8
Saved metrics: /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/evalua

,defense_id,attack_id,num_samples,clean_accuracy,robust_accuracy,attack_success_rate,mean_l2_successful,attack_time_seconds,defense_time_seconds,total_time_seconds
0,rand_smooth,cw_l2,10000,0.7515,0.7095,0.068263,0.317013,251.912653,102.686933,355.329976
1,rand_smooth,deepfool,10000,0.7515,0.7149,0.075449,0.460774,710.565312,102.377498,815.157337


In [19]:
assert RS_FINAL_METRICS_PATH.is_file()

verified_rs_final_df = pd.read_csv(
    RS_FINAL_METRICS_PATH
)

assert list(
    verified_rs_final_df.columns
) == CANONICAL_METRIC_COLUMNS

assert len(verified_rs_final_df) == 2

assert verified_rs_final_df[
    "defense_id"
].eq("rand_smooth").all()

assert set(
    verified_rs_final_df["attack_id"]
) == {"deepfool", "cw_l2"}

assert verified_rs_final_df[
    "num_samples"
].eq(10_000).all()

assert verified_rs_final_df[
    "clean_accuracy"
].nunique() == 1

for accuracy_column in (
    "clean_accuracy",
    "robust_accuracy",
    "attack_success_rate",
):
    assert verified_rs_final_df[
        accuracy_column
    ].between(0.0, 1.0).all()

assert verified_rs_final_df[
    "epsilon"
].isna().all()

assert verified_rs_final_df[
    "alpha"
].isna().all()


for attack_id in FINAL_RS_RUNS:
    examples_path = (
        RS_FINAL_EXAMPLES_DIRECTORY
        / (
            f"rand_smooth__"
            f"{attack_id}__examples.pt"
        )
    )

    assert examples_path.is_file()

    examples = torch.load(
        examples_path,
        map_location="cpu",
        weights_only=False,
    )

    assert examples["originals"] is not None
    assert examples["adversarials"] is not None
    assert examples["labels"] is not None

    assert (
        examples["originals"].shape
        == examples["adversarials"].shape
    )

    assert examples["labels"].shape[0] == 8

    assert torch.all(
        examples[
            "clean_vote_counts"
        ].sum(dim=1)
        == SMOOTH_NUM_SAMPLES
    )

    assert torch.all(
        examples[
            "adversarial_vote_counts"
        ].sum(dim=1)
        == SMOOTH_NUM_SAMPLES
    )

    print(
        "VERIFIED:",
        examples_path.name,
        "successful examples:",
        examples["labels"].shape[0],
    )


rs_final_metrics_hash = hashlib.sha256(
    RS_FINAL_METRICS_PATH.read_bytes()
).hexdigest()

print(
    "\nRandomized-smoothing metrics SHA256:",
    rs_final_metrics_hash,
)


STANDARD_FINAL_METRICS_PATH = (
    DRIVE_RESULTS_DIR
    / "evaluate_colab_expensive_standard_final.csv"
)

assert STANDARD_FINAL_METRICS_PATH.is_file()

standard_final_hash = hashlib.sha256(
    STANDARD_FINAL_METRICS_PATH.read_bytes()
).hexdigest()

assert standard_final_hash == (
    "68a164455dc0e271efeaa3ef58a158ccd9"
    "bd4eff5e03bd01fb1ed89ddd46ab36"
)

standard_final_df = pd.read_csv(
    STANDARD_FINAL_METRICS_PATH
)

combined_expensive_df = pd.concat(
    [
        standard_final_df,
        verified_rs_final_df,
    ],
    ignore_index=True,
)

display(
    combined_expensive_df[
        [
            "defense_id",
            "attack_id",
            "num_samples",
            "clean_accuracy",
            "robust_accuracy",
            "attack_success_rate",
            "mean_l2_successful",
            "attack_time_seconds",
            "defense_time_seconds",
            "total_time_seconds",
        ]
    ].sort_values(
        ["defense_id", "attack_id"]
    )
)

print(
    "\nALL FINAL RANDOMIZED-SMOOTHING "
    "DEEPFOOL/CW EVALUATIONS PASSED."
)

VERIFIED: rand_smooth__deepfool__examples.pt successful examples: 8
VERIFIED: rand_smooth__cw_l2__examples.pt successful examples: 8

Randomized-smoothing metrics SHA256: f3c6ed39604a31cdb4c568efdf6411f54bb116c3dde603cbd427e2126410d184


,defense_id,attack_id,num_samples,clean_accuracy,robust_accuracy,attack_success_rate,mean_l2_successful,attack_time_seconds,defense_time_seconds,total_time_seconds
0,none,cw_l2,10000,0.8403,0.0000,1.000000,0.186184,247.555366,0.000000,250.504518
1,none,deepfool,10000,0.8403,0.0982,1.000000,0.117538,823.553014,0.000000,835.113201
2,pgd_at,cw_l2,10000,0.6798,0.3565,0.475581,0.523592,249.479599,0.000000,252.598311
3,pgd_at,deepfool,10000,0.6798,0.1473,0.999853,1.262114,774.959111,0.000000,786.611291
4,rand_smooth,cw_l2,10000,0.7515,0.7095,0.068263,0.317013,251.912653,102.686933,355.329976
5,rand_smooth,deepfool,10000,0.7515,0.7149,0.075449,0.460774,710.565312,102.377498,815.157337



ALL FINAL RANDOMIZED-SMOOTHING DEEPFOOL/CW EVALUATIONS PASSED.


In [20]:
from src.attacks.fgsm import fgsm_attack
from src.attacks.pgd import pgd_attack


FGSM_EPSILON_PIXELS = (
    2,
    4,
    8,
    16,
)

BOUNDED_RUN_CONFIGS = [
    {
        "attack_id": "fgsm",
        "epsilon_pixels": epsilon_pixels,
        "epsilon": epsilon_pixels / 255,
        "alpha": np.nan,
        "attack_steps": 1,
    }
    for epsilon_pixels in FGSM_EPSILON_PIXELS
]

BOUNDED_RUN_CONFIGS.append(
    {
        "attack_id": "pgd",
        "epsilon_pixels": 8,
        "epsilon": 8 / 255,
        "alpha": 2 / 255,
        "attack_steps": 20,
    }
)


def make_bounded_run_id(
    defense_id,
    config,
):
    attack_id = config["attack_id"]
    epsilon_pixels = config[
        "epsilon_pixels"
    ]

    if attack_id == "fgsm":
        parameter_text = (
            f"eps{epsilon_pixels}"
        )
    else:
        parameter_text = (
            "eps8__alpha2__steps20"
        )

    if defense_id == "rand_smooth":
        parameter_text += (
            "__sigma0p25__votes100"
        )

    return (
        f"{defense_id}__{attack_id}__"
        f"{parameter_text}__seed{SEED}"
    )


def bounded_examples_path(
    defense_id,
    config,
):
    run_id = make_bounded_run_id(
        defense_id,
        config,
    )

    return (
        DRIVE_RESULTS_DIR
        / "final_attack_examples"
        / f"{run_id}__examples.pt"
    )


def reset_bounded_attack_seed():
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(SEED)


def evaluate_bounded_attack(
    defense_id,
    config,
):
    if defense_id not in (
        "none",
        "pgd_at",
        "rand_smooth",
    ):
        raise ValueError(
            f"Unsupported defense: {defense_id}"
        )

    attack_id = config["attack_id"]
    epsilon = config["epsilon"]
    alpha = config["alpha"]
    attack_steps = config[
        "attack_steps"
    ]

    model = CLASSIFIERS[defense_id]
    model.eval()

    use_smoothing = (
        defense_id == "rand_smooth"
    )

    batch_size = (
        64 if use_smoothing else 128
    )

    loader = DataLoader(
        TEST_DATASET,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=DEVICE.type == "cuda",
    )

    total_samples = 0
    clean_correct_total = 0
    adversarial_correct_total = 0
    successful_total = 0

    l2_sum = 0.0
    linf_sum = 0.0
    successful_l2_sum = 0.0
    successful_linf_sum = 0.0

    attack_time = 0.0
    adversarial_defense_time = 0.0
    cache_offset = 0

    saved_examples = {
        "originals": [],
        "adversarials": [],
        "labels": [],
        "clean_logits": [],
        "adversarial_logits": [],
        "clean_predictions": [],
        "adversarial_predictions": [],
        "clean_vote_counts": [],
        "adversarial_vote_counts": [],
    }

    if use_smoothing:
        assert rs_clean_cache_is_valid(
            rs_final_clean_cache
        )

        smoothing_generator = (
            torch.Generator(
                device=DEVICE
            ).manual_seed(SEED)
        )
    else:
        smoothing_generator = None

    reset_bounded_attack_seed()

    synchronize_device()
    evaluation_start = time.perf_counter()

    for images, labels in loader:
        images = images.to(
            DEVICE,
            non_blocking=True,
        )
        labels = labels.to(
            DEVICE,
            non_blocking=True,
        )

        batch_samples = images.shape[0]

        if use_smoothing:
            cache_slice = slice(
                cache_offset,
                cache_offset + batch_samples,
            )

            cached_labels = (
                rs_final_clean_cache[
                    "labels"
                ][cache_slice].to(DEVICE)
            )

            assert torch.equal(
                labels,
                cached_labels,
            )

            clean_predictions = (
                rs_final_clean_cache[
                    "predictions"
                ][cache_slice].to(DEVICE)
            )

            clean_vote_counts = (
                rs_final_clean_cache[
                    "vote_counts"
                ][cache_slice].to(DEVICE)
            )

            clean_logits = None

        else:
            with torch.no_grad():
                clean_logits = model(images)

            clean_predictions = (
                clean_logits.argmax(dim=1)
            )
            clean_vote_counts = None

        synchronize_device()
        attack_start = time.perf_counter()

        if attack_id == "fgsm":
            adversarial_images = fgsm_attack(
                model=model,
                images=images,
                labels=labels,
                epsilon=epsilon,
            )

        elif attack_id == "pgd":
            adversarial_images = pgd_attack(
                model=model,
                images=images,
                labels=labels,
                epsilon=epsilon,
                alpha=alpha,
                steps=attack_steps,
                random_start=True,
                restarts=1,
            )

        else:
            raise ValueError(
                f"Unsupported attack: {attack_id}"
            )

        synchronize_device()
        attack_time += (
            time.perf_counter() - attack_start
        )

        assert adversarial_images.shape == images.shape
        assert adversarial_images.dtype == images.dtype
        assert adversarial_images.device == images.device
        assert adversarial_images.min().item() >= 0.0
        assert adversarial_images.max().item() <= 1.0

        perturbation = (
            adversarial_images - images
        ).flatten(1)

        l2_norms = perturbation.norm(
            p=2,
            dim=1,
        )

        linf_norms = perturbation.abs().amax(
            dim=1,
        )

        assert (
            linf_norms.max().item()
            <= epsilon + 1e-6
        )

        if use_smoothing:
            synchronize_device()
            defense_start = time.perf_counter()

            (
                adversarial_predictions,
                adversarial_vote_counts,
            ) = predict_smoothed(
                model=model,
                images=adversarial_images,
                sigma=SMOOTH_SIGMA,
                num_samples=SMOOTH_NUM_SAMPLES,
                noise_batch_size=(
                    SMOOTH_NOISE_BATCH_SIZE
                ),
                num_classes=SMOOTH_NUM_CLASSES,
                clip_noise=True,
                generator=smoothing_generator,
            )

            synchronize_device()
            adversarial_defense_time += (
                time.perf_counter()
                - defense_start
            )

            assert torch.all(
                adversarial_vote_counts.sum(
                    dim=1
                )
                == SMOOTH_NUM_SAMPLES
            )

            adversarial_logits = None

        else:
            with torch.no_grad():
                adversarial_logits = model(
                    adversarial_images
                )

            adversarial_predictions = (
                adversarial_logits.argmax(
                    dim=1
                )
            )
            adversarial_vote_counts = None

        clean_correct = (
            clean_predictions.eq(labels)
        )

        adversarial_correct = (
            adversarial_predictions.eq(labels)
        )

        successful = (
            clean_correct
            & (~adversarial_correct)
        )

        total_samples += batch_samples
        clean_correct_total += (
            clean_correct.sum().item()
        )
        adversarial_correct_total += (
            adversarial_correct.sum().item()
        )
        successful_total += (
            successful.sum().item()
        )

        l2_sum += l2_norms.sum().item()
        linf_sum += linf_norms.sum().item()

        if bool(successful.any()):
            successful_l2_sum += (
                l2_norms[successful]
                .sum()
                .item()
            )

            successful_linf_sum += (
                linf_norms[successful]
                .sum()
                .item()
            )

            already_saved = sum(
                tensor.shape[0]
                for tensor in saved_examples[
                    "labels"
                ]
            )

            remaining = max(
                8 - already_saved,
                0,
            )

            if remaining > 0:
                selected = torch.where(
                    successful
                )[0][:remaining]

                if use_smoothing:
                    with torch.no_grad():
                        selected_clean_logits = (
                            model(
                                images[selected]
                            )
                        )

                        selected_adv_logits = (
                            model(
                                adversarial_images[
                                    selected
                                ]
                            )
                        )
                else:
                    selected_clean_logits = (
                        clean_logits[selected]
                    )

                    selected_adv_logits = (
                        adversarial_logits[
                            selected
                        ]
                    )

                saved_examples[
                    "originals"
                ].append(
                    images[
                        selected
                    ].detach().cpu()
                )

                saved_examples[
                    "adversarials"
                ].append(
                    adversarial_images[
                        selected
                    ].detach().cpu()
                )

                saved_examples[
                    "labels"
                ].append(
                    labels[
                        selected
                    ].detach().cpu()
                )

                saved_examples[
                    "clean_logits"
                ].append(
                    selected_clean_logits
                    .detach()
                    .cpu()
                )

                saved_examples[
                    "adversarial_logits"
                ].append(
                    selected_adv_logits
                    .detach()
                    .cpu()
                )

                saved_examples[
                    "clean_predictions"
                ].append(
                    clean_predictions[
                        selected
                    ].detach().cpu()
                )

                saved_examples[
                    "adversarial_predictions"
                ].append(
                    adversarial_predictions[
                        selected
                    ].detach().cpu()
                )

                if use_smoothing:
                    saved_examples[
                        "clean_vote_counts"
                    ].append(
                        clean_vote_counts[
                            selected
                        ].detach().cpu()
                    )

                    saved_examples[
                        "adversarial_vote_counts"
                    ].append(
                        adversarial_vote_counts[
                            selected
                        ].detach().cpu()
                    )

        cache_offset += batch_samples

    synchronize_device()
    evaluation_time = (
        time.perf_counter()
        - evaluation_start
    )

    assert total_samples == len(
        TEST_DATASET
    )

    if use_smoothing:
        assert cache_offset == len(
            TEST_DATASET
        )

        clean_defense_time = (
            rs_final_clean_cache[
                "defense_time_seconds"
            ]
        )
    else:
        clean_defense_time = 0.0

    defense_time = (
        clean_defense_time
        + adversarial_defense_time
    )

    total_time = (
        evaluation_time
        + clean_defense_time
    )

    attack_success_rate = (
        successful_total
        / clean_correct_total
        if clean_correct_total > 0
        else float("nan")
    )

    mean_l2_successful = (
        successful_l2_sum
        / successful_total
        if successful_total > 0
        else float("nan")
    )

    mean_linf_successful = (
        successful_linf_sum
        / successful_total
        if successful_total > 0
        else float("nan")
    )

    row = {
        "run_id": make_bounded_run_id(
            defense_id,
            config,
        ),
        "model_id": "resnet20",
        "defense_id": defense_id,
        "attack_id": attack_id,
        "split": "test",
        "num_samples": total_samples,
        "seed": SEED,
        "epsilon": epsilon,
        "alpha": (
            alpha
            if attack_id == "pgd"
            else np.nan
        ),
        "attack_steps": attack_steps,
        "clean_accuracy": (
            clean_correct_total
            / total_samples
        ),
        "robust_accuracy": (
            adversarial_correct_total
            / total_samples
        ),
        "attack_success_rate": (
            attack_success_rate
        ),
        "mean_l2": (
            l2_sum / total_samples
        ),
        "mean_l2_successful": (
            mean_l2_successful
        ),
        "mean_linf": (
            linf_sum / total_samples
        ),
        "mean_linf_successful": (
            mean_linf_successful
        ),
        "attack_time_seconds": attack_time,
        "defense_time_seconds": (
            defense_time
        ),
        "total_time_seconds": total_time,
        "checkpoint_name": (
            CHECKPOINT_NAMES[defense_id]
        ),
    }

    concatenated_examples = {}

    for key, tensors in saved_examples.items():
        concatenated_examples[key] = (
            torch.cat(tensors, dim=0)
            if tensors
            else None
        )

    return row, concatenated_examples


print(
    "FGSM/PGD FINAL EVALUATOR DEFINED."
)

FGSM/PGD FINAL EVALUATOR DEFINED.


In [21]:
BOUNDED_FINAL_METRICS_PATH = (
    DRIVE_RESULTS_DIR
    / "evaluate_colab_bounded_final.csv"
)

BOUNDED_EXAMPLES_DIRECTORY = (
    DRIVE_RESULTS_DIR
    / "final_attack_examples"
)

BOUNDED_EXAMPLES_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


if BOUNDED_FINAL_METRICS_PATH.is_file():
    bounded_final_df = pd.read_csv(
        BOUNDED_FINAL_METRICS_PATH
    )

    assert list(
        bounded_final_df.columns
    ) == CANONICAL_METRIC_COLUMNS

    print(
        "Existing bounded results loaded:",
        len(bounded_final_df),
    )

else:
    bounded_final_df = pd.DataFrame(
        columns=CANONICAL_METRIC_COLUMNS
    )

    print(
        "No previous FGSM/PGD "
        "final results found."
    )


def bounded_run_is_complete(
    metrics_df,
    defense_id,
    config,
):
    run_id = make_bounded_run_id(
        defense_id,
        config,
    )

    examples_path = bounded_examples_path(
        defense_id,
        config,
    )

    if metrics_df.empty:
        return False

    selected = metrics_df.loc[
        metrics_df["run_id"].eq(run_id)
    ]

    if len(selected) != 1:
        return False

    row = selected.iloc[0]

    return (
        row["num_samples"] == 10_000
        and pd.notna(
            row["robust_accuracy"]
        )
        and pd.notna(
            row["attack_success_rate"]
        )
        and examples_path.is_file()
    )


for defense_id in (
    "none",
    "pgd_at",
    "rand_smooth",
):
    for config in BOUNDED_RUN_CONFIGS:
        run_id = make_bounded_run_id(
            defense_id,
            config,
        )

        if bounded_run_is_complete(
            bounded_final_df,
            defense_id,
            config,
        ):
            print("=" * 70)
            print(
                "SKIPPED — already completed:",
                run_id,
            )
            continue

        print("=" * 70)
        print("FINAL RUN STARTED:", run_id)
        print("Samples:", len(TEST_DATASET))

        row, examples = (
            evaluate_bounded_attack(
                defense_id=defense_id,
                config=config,
            )
        )

        examples_path = (
            bounded_examples_path(
                defense_id,
                config,
            )
        )

        temporary_examples_path = (
            examples_path.with_name(
                examples_path.name + ".tmp"
            )
        )

        torch.save(
            examples,
            temporary_examples_path,
        )

        temporary_examples_path.replace(
            examples_path
        )

        if not bounded_final_df.empty:
            bounded_final_df = (
                bounded_final_df.loc[
                    ~bounded_final_df[
                        "run_id"
                    ].eq(run_id)
                ].copy()
            )

        new_row_df = pd.DataFrame(
            [
                {
                    column: row[column]
                    for column
                    in CANONICAL_METRIC_COLUMNS
                }
            ],
            columns=CANONICAL_METRIC_COLUMNS,
        )

        if bounded_final_df.empty:
            bounded_final_df = new_row_df
        else:
            bounded_final_df = pd.concat(
                [
                    bounded_final_df,
                    new_row_df,
                ],
                ignore_index=True,
            )

        bounded_final_df = (
            bounded_final_df[
                CANONICAL_METRIC_COLUMNS
            ]
            .sort_values(
                [
                    "defense_id",
                    "attack_id",
                    "epsilon",
                ]
            )
            .reset_index(drop=True)
        )

        temporary_metrics_path = (
            BOUNDED_FINAL_METRICS_PATH
            .with_name(
                BOUNDED_FINAL_METRICS_PATH.name
                + ".tmp"
            )
        )

        bounded_final_df.to_csv(
            temporary_metrics_path,
            index=False,
        )

        temporary_metrics_path.replace(
            BOUNDED_FINAL_METRICS_PATH
        )

        current_hash = hashlib.sha256(
            BOUNDED_FINAL_METRICS_PATH
            .read_bytes()
        ).hexdigest()

        successful_examples = (
            0
            if examples["labels"] is None
            else examples["labels"].shape[0]
        )

        print(
            f"Clean accuracy: "
            f"{100 * row['clean_accuracy']:.2f}%"
        )
        print(
            f"Robust accuracy: "
            f"{100 * row['robust_accuracy']:.2f}%"
        )
        print(
            f"Attack success rate: "
            f"{100 * row['attack_success_rate']:.2f}%"
        )
        print(
            f"Mean Linf successful: "
            f"{row['mean_linf_successful']:.6f}"
        )
        print(
            f"Attack time: "
            f"{row['attack_time_seconds']:.2f}s"
        )
        print(
            f"Defense time: "
            f"{row['defense_time_seconds']:.2f}s"
        )
        print(
            f"Total time: "
            f"{row['total_time_seconds']:.2f}s"
        )
        print(
            "Successful examples saved:",
            successful_examples,
        )
        print("Saved metrics:", BOUNDED_FINAL_METRICS_PATH)
        print("Saved examples:", examples_path)
        print("Metrics SHA256:", current_hash)

        gc.collect()

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()


print("\nCURRENT FGSM/PGD RESULTS:")

display(
    bounded_final_df[
        [
            "defense_id",
            "attack_id",
            "epsilon",
            "alpha",
            "attack_steps",
            "clean_accuracy",
            "robust_accuracy",
            "attack_success_rate",
            "mean_linf_successful",
            "attack_time_seconds",
            "defense_time_seconds",
            "total_time_seconds",
        ]
    ]
)

No previous FGSM/PGD final results found.
FINAL RUN STARTED: none__fgsm__eps2__seed42
Samples: 10000
Clean accuracy: 84.03%
Robust accuracy: 14.92%
Attack success rate: 82.27%
Mean Linf successful: 0.007843
Attack time: 1.45s
Defense time: 0.00s
Total time: 2.79s
Successful examples saved: 8
Saved metrics: /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/evaluate_colab_bounded_final.csv
Saved examples: /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/final_attack_examples/none__fgsm__eps2__seed42__examples.pt
Metrics SHA256: 848aee905044a54439ae1ec9e7767046b7dbbc3a9097a2ce5c1e58f32df20ad9
FINAL RUN STARTED: none__fgsm__eps4__seed42
Samples: 10000
Clean accuracy: 84.03%
Robust accuracy: 7.05%
Attack success rate: 91.63%
Mean Linf successful: 0.015686
Attack time: 1.62s
Defense time: 0.00s
Total time: 3.10s
Successful examples saved: 8
Saved metrics: /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/evaluate_colab_bounded_final.c

,defense_id,attack_id,epsilon,alpha,attack_steps,clean_accuracy,robust_accuracy,attack_success_rate,mean_linf_successful,attack_time_seconds,defense_time_seconds,total_time_seconds
0,none,fgsm,0.007843,NaN,1,0.8403,0.1492,0.822682,0.007843,1.446744,0.000000,2.788243
1,none,fgsm,0.015686,NaN,1,0.8403,0.0705,0.916339,0.015686,1.622644,0.000000,3.100188
2,none,fgsm,0.031373,NaN,1,0.8403,0.0473,0.943830,0.031373,1.653864,0.000000,3.208209
3,none,fgsm,0.062745,NaN,1,0.8403,0.0562,0.936808,0.062745,1.349923,0.000000,2.657154
4,none,pgd,0.031373,0.007843,20,0.8403,0.0000,1.000000,0.031373,27.522206,0.000000,28.872413
5,pgd_at,fgsm,0.007843,NaN,1,0.6798,0.6152,0.095028,0.007843,1.360046,0.000000,2.675085
6,pgd_at,fgsm,0.015686,NaN,1,0.6798,0.5546,0.184172,0.015686,1.619238,0.000000,3.107745
7,pgd_at,fgsm,0.031373,NaN,1,0.6798,0.4357,0.359076,0.031373,1.694778,0.000000,3.327541
8,pgd_at,fgsm,0.062745,NaN,1,0.6798,0.2628,0.613416,0.062745,1.350947,0.000000,2.673762
9,pgd_at,pgd,0.031373,0.007843,20,0.6798,0.3964,0.417034,0.031373,26.861545,0.000000,28.167505


In [22]:
assert BOUNDED_FINAL_METRICS_PATH.is_file()

verified_bounded_df = pd.read_csv(
    BOUNDED_FINAL_METRICS_PATH
)

assert list(
    verified_bounded_df.columns
) == CANONICAL_METRIC_COLUMNS

assert len(verified_bounded_df) == 15

assert set(
    verified_bounded_df["defense_id"]
) == {
    "none",
    "pgd_at",
    "rand_smooth",
}

assert set(
    verified_bounded_df["attack_id"]
) == {
    "fgsm",
    "pgd",
}

assert verified_bounded_df[
    "num_samples"
].eq(10_000).all()


for defense_id in (
    "none",
    "pgd_at",
    "rand_smooth",
):
    defense_rows = (
        verified_bounded_df.loc[
            verified_bounded_df[
                "defense_id"
            ].eq(defense_id)
        ]
    )

    fgsm_rows = defense_rows.loc[
        defense_rows["attack_id"].eq(
            "fgsm"
        )
    ]

    pgd_rows = defense_rows.loc[
        defense_rows["attack_id"].eq(
            "pgd"
        )
    ]

    assert len(fgsm_rows) == 4
    assert len(pgd_rows) == 1

    assert np.allclose(
        sorted(
            fgsm_rows["epsilon"].tolist()
        ),
        sorted(
            [
                2 / 255,
                4 / 255,
                8 / 255,
                16 / 255,
            ]
        ),
    )

    assert fgsm_rows[
        "alpha"
    ].isna().all()

    assert fgsm_rows[
        "attack_steps"
    ].eq(1).all()

    assert np.isclose(
        pgd_rows.iloc[0]["epsilon"],
        8 / 255,
    )

    assert np.isclose(
        pgd_rows.iloc[0]["alpha"],
        2 / 255,
    )

    assert (
        pgd_rows.iloc[0][
            "attack_steps"
        ]
        == 20
    )


for accuracy_column in (
    "clean_accuracy",
    "robust_accuracy",
    "attack_success_rate",
):
    assert verified_bounded_df[
        accuracy_column
    ].between(0.0, 1.0).all()


expected_run_ids = set()

for defense_id in (
    "none",
    "pgd_at",
    "rand_smooth",
):
    for config in BOUNDED_RUN_CONFIGS:
        run_id = make_bounded_run_id(
            defense_id,
            config,
        )

        expected_run_ids.add(run_id)

        examples_path = (
            bounded_examples_path(
                defense_id,
                config,
            )
        )

        assert examples_path.is_file()

        examples = torch.load(
            examples_path,
            map_location="cpu",
            weights_only=False,
        )

        assert examples["originals"] is not None
        assert examples["adversarials"] is not None
        assert examples["labels"] is not None

        assert examples["labels"].shape[0] == 8

        assert (
            examples["originals"].shape
            == examples[
                "adversarials"
            ].shape
        )

        perturbations = (
            examples["adversarials"]
            - examples["originals"]
        ).flatten(1)

        maximum_linf = (
            perturbations.abs()
            .amax(dim=1)
            .max()
            .item()
        )

        assert (
            maximum_linf
            <= config["epsilon"] + 1e-6
        )

        if defense_id == "rand_smooth":
            assert torch.all(
                examples[
                    "clean_vote_counts"
                ].sum(dim=1)
                == SMOOTH_NUM_SAMPLES
            )

            assert torch.all(
                examples[
                    "adversarial_vote_counts"
                ].sum(dim=1)
                == SMOOTH_NUM_SAMPLES
            )

        print(
            "VERIFIED:",
            examples_path.name,
            "successful examples:",
            examples["labels"].shape[0],
        )


assert set(
    verified_bounded_df["run_id"]
) == expected_run_ids

bounded_final_hash = hashlib.sha256(
    BOUNDED_FINAL_METRICS_PATH
    .read_bytes()
).hexdigest()

print(
    "\nBounded metrics SHA256:",
    bounded_final_hash,
)


standard_expensive_path = (
    DRIVE_RESULTS_DIR
    / "evaluate_colab_expensive_standard_final.csv"
)

rand_smooth_expensive_path = (
    DRIVE_RESULTS_DIR
    / "evaluate_colab_expensive_rand_smooth_final.csv"
)

assert hashlib.sha256(
    standard_expensive_path.read_bytes()
).hexdigest() == (
    "68a164455dc0e271efeaa3ef58a158ccd9"
    "bd4eff5e03bd01fb1ed89ddd46ab36"
)

assert hashlib.sha256(
    rand_smooth_expensive_path.read_bytes()
).hexdigest() == (
    "f3c6ed39604a31cdb4c568efdf6411f54"
    "bb116c3dde603cbd427e2126410d184"
)

all_completed_df = pd.concat(
    [
        pd.read_csv(
            standard_expensive_path
        ),
        pd.read_csv(
            rand_smooth_expensive_path
        ),
        verified_bounded_df,
    ],
    ignore_index=True,
)

assert len(all_completed_df) == 21
assert all_completed_df[
    "run_id"
].is_unique

display(
    verified_bounded_df[
        [
            "defense_id",
            "attack_id",
            "epsilon",
            "clean_accuracy",
            "robust_accuracy",
            "attack_success_rate",
            "mean_linf_successful",
            "attack_time_seconds",
            "defense_time_seconds",
            "total_time_seconds",
        ]
    ].sort_values(
        [
            "defense_id",
            "attack_id",
            "epsilon",
        ]
    )
)

print(
    "\nTOTAL VERIFIED ATTACK RUNS:",
    len(all_completed_df),
)

print(
    "\nALL FINAL FGSM/PGD "
    "EVALUATIONS PASSED."
)

VERIFIED: none__fgsm__eps2__seed42__examples.pt successful examples: 8
VERIFIED: none__fgsm__eps4__seed42__examples.pt successful examples: 8
VERIFIED: none__fgsm__eps8__seed42__examples.pt successful examples: 8
VERIFIED: none__fgsm__eps16__seed42__examples.pt successful examples: 8
VERIFIED: none__pgd__eps8__alpha2__steps20__seed42__examples.pt successful examples: 8
VERIFIED: pgd_at__fgsm__eps2__seed42__examples.pt successful examples: 8
VERIFIED: pgd_at__fgsm__eps4__seed42__examples.pt successful examples: 8
VERIFIED: pgd_at__fgsm__eps8__seed42__examples.pt successful examples: 8
VERIFIED: pgd_at__fgsm__eps16__seed42__examples.pt successful examples: 8
VERIFIED: pgd_at__pgd__eps8__alpha2__steps20__seed42__examples.pt successful examples: 8
VERIFIED: rand_smooth__fgsm__eps2__sigma0p25__votes100__seed42__examples.pt successful examples: 8
VERIFIED: rand_smooth__fgsm__eps4__sigma0p25__votes100__seed42__examples.pt successful examples: 8
VERIFIED: rand_smooth__fgsm__eps8__sigma0p25__vo

,defense_id,attack_id,epsilon,clean_accuracy,robust_accuracy,attack_success_rate,mean_linf_successful,attack_time_seconds,defense_time_seconds,total_time_seconds
0,none,fgsm,0.007843,0.8403,0.1492,0.822682,0.007843,1.446744,0.000000,2.788243
1,none,fgsm,0.015686,0.8403,0.0705,0.916339,0.015686,1.622644,0.000000,3.100188
2,none,fgsm,0.031373,0.8403,0.0473,0.943830,0.031373,1.653864,0.000000,3.208209
3,none,fgsm,0.062745,0.8403,0.0562,0.936808,0.062745,1.349923,0.000000,2.657154
4,none,pgd,0.031373,0.8403,0.0000,1.000000,0.031373,27.522206,0.000000,28.872413
5,pgd_at,fgsm,0.007843,0.6798,0.6152,0.095028,0.007843,1.360046,0.000000,2.675085
6,pgd_at,fgsm,0.015686,0.6798,0.5546,0.184172,0.015686,1.619238,0.000000,3.107745
7,pgd_at,fgsm,0.031373,0.6798,0.4357,0.359076,0.031373,1.694778,0.000000,3.327541
8,pgd_at,fgsm,0.062745,0.6798,0.2628,0.613416,0.062745,1.350947,0.000000,2.673762
9,pgd_at,pgd,0.031373,0.6798,0.3964,0.417034,0.031373,26.861545,0.000000,28.167505



TOTAL VERIFIED ATTACK RUNS: 21

ALL FINAL FGSM/PGD EVALUATIONS PASSED.


In [23]:
from pathlib import Path

import ast
import hashlib
import importlib
import importlib.metadata
import inspect
import sys
import traceback


DIFFUSION_SOURCE_PATH = (
    PROJECT_ROOT
    / "src"
    / "defenses"
    / "diffusion_purification.py"
)

assert DIFFUSION_SOURCE_PATH.is_file(), (
    DIFFUSION_SOURCE_PATH
)

diffusion_source = (
    DIFFUSION_SOURCE_PATH.read_text(
        encoding="utf-8"
    )
)

# بررسی صحت نحوی بدون اجرای importها
compile(
    diffusion_source,
    str(DIFFUSION_SOURCE_PATH),
    "exec",
)

diffusion_hash = hashlib.sha256(
    DIFFUSION_SOURCE_PATH.read_bytes()
).hexdigest()

print("File:", DIFFUSION_SOURCE_PATH)
print(
    "Lines:",
    len(diffusion_source.splitlines()),
)
print("SHA256:", diffusion_hash)
print("PYTHON SYNTAX PASSED.")


# --------------------------------------------------
# نسخه کتابخانه‌های مرتبط
# --------------------------------------------------

print("\nPACKAGE VERSIONS:")

for package_name in (
    "torch",
    "diffusers",
    "transformers",
    "accelerate",
    "huggingface-hub",
    "safetensors",
):
    try:
        version = (
            importlib.metadata.version(
                package_name
            )
        )
    except (
        importlib.metadata.PackageNotFoundError
    ):
        version = "NOT INSTALLED"

    print(
        f"- {package_name}: {version}"
    )


# --------------------------------------------------
# imports و تعریف‌های سطح ماژول از طریق AST
# --------------------------------------------------

syntax_tree = ast.parse(
    diffusion_source,
    filename=str(DIFFUSION_SOURCE_PATH),
)

source_lines = diffusion_source.splitlines()


print("\nMODULE IMPORTS:")

for node in syntax_tree.body:
    if isinstance(
        node,
        (ast.Import, ast.ImportFrom),
    ):
        import_text = ast.get_source_segment(
            diffusion_source,
            node,
        )
        print(import_text)


print("\nTOP-LEVEL DEFINITIONS:")

for node in syntax_tree.body:
    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
            ast.ClassDef,
        ),
    ):
        first_body_line = (
            node.body[0].lineno
            if node.body
            else node.end_lineno
        )

        header_lines = source_lines[
            node.lineno - 1:
            first_body_line - 1
        ]

        print("-" * 60)
        print(
            "\n".join(header_lines).rstrip()
        )


# --------------------------------------------------
# تلاش برای import واقعی ماژول
# --------------------------------------------------

print("\nREAL MODULE IMPORT:")

module_name = (
    "src.defenses.diffusion_purification"
)

importlib.invalidate_caches()
sys.modules.pop(module_name, None)

try:
    diffusion_module = (
        importlib.import_module(
            module_name
        )
    )

    print("MODULE IMPORT PASSED.")
    print(
        "__all__:",
        getattr(
            diffusion_module,
            "__all__",
            None,
        ),
    )

    print("\nCALLABLE SIGNATURES:")

    for name, value in vars(
        diffusion_module
    ).items():
        if (
            not name.startswith("_")
            and callable(value)
            and getattr(
                value,
                "__module__",
                None,
            ) == module_name
        ):
            print(
                f"{name}"
                f"{inspect.signature(value)}"
            )

    for public_name in (
        "load_diffusion_components",
        "purify_images",
    ):
        if hasattr(
            diffusion_module,
            public_name,
        ):
            print(
                "\n"
                + "=" * 70
            )
            print(
                f"SOURCE: {public_name}"
            )
            print("=" * 70)
            print(
                inspect.getsource(
                    getattr(
                        diffusion_module,
                        public_name,
                    )
                )
            )

except Exception:
    print("MODULE IMPORT FAILED:")
    traceback.print_exc(limit=4)


# --------------------------------------------------
# بررسی requirements
# --------------------------------------------------

requirements_path = (
    PROJECT_ROOT / "requirements.txt"
)

print("\nRELATED REQUIREMENTS:")

if requirements_path.is_file():
    related_lines = [
        line
        for line in requirements_path.read_text(
            encoding="utf-8"
        ).splitlines()
        if any(
            keyword in line.lower()
            for keyword in (
                "diffuser",
                "transformer",
                "accelerate",
                "huggingface",
                "safetensor",
            )
        )
    ]

    if related_lines:
        for line in related_lines:
            print("-", line)
    else:
        print(
            "No diffusion-related dependency "
            "was found in requirements.txt."
        )
else:
    print("requirements.txt not found.")


print(
    "\nDIFFUSION SOURCE INSPECTION COMPLETED."
)

File: /content/AI-Project-Adversarial-Attack-Defense/src/defenses/diffusion_purification.py
Lines: 878
SHA256: 75708d53c4bb616de6ccf85392dd58307368ec73d118bedfadaf017e10ca622a
PYTHON SYNTAX PASSED.

PACKAGE VERSIONS:
- torch: 2.11.0+cu128
- diffusers: 0.39.0
- transformers: 5.13.1
- accelerate: 1.14.0
- huggingface-hub: 1.23.0
- safetensors: 0.8.0

MODULE IMPORTS:
from __future__ import annotations
from collections.abc import Mapping
from typing import Any, Optional
import torch
from torch import Tensor, nn

TOP-LEVEL DEFINITIONS:
------------------------------------------------------------
def _config_value(
    config: Any,
    key: str,
    default: Any = None,
) -> Any:
------------------------------------------------------------
def _module_device_and_dtype(
    module: nn.Module,
) -> tuple[torch.device, torch.dtype]:
------------------------------------------------------------
def _validate_images(
    images: Tensor,
) -> None:
--------------------------------------------------

In [24]:
import hashlib
import importlib.metadata
import json
import os
import time

from src.defenses.diffusion_purification import (
    DEFAULT_DIFFUSION_MODEL_ID,
    load_diffusion_components,
    purify_images,
)


DIFFUSION_MODEL_ID = (
    "google/ddpm-cifar10-32"
)

DIFFUSION_TIMESTEP = 50
DIFFUSION_DTYPE = torch.float32


# cache مدل روی Drive باقی می‌ماند تا پس از
# قطع Runtime دوباره دانلود نشود.
DIFFUSION_CACHE_ROOT = (
    DRIVE_PROJECT_ROOT
    / "huggingface_cache"
)

DIFFUSION_CACHE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

os.environ["HF_HOME"] = str(
    DIFFUSION_CACHE_ROOT
)

os.environ["HUGGINGFACE_HUB_CACHE"] = str(
    DIFFUSION_CACHE_ROOT / "hub"
)


assert (
    DEFAULT_DIFFUSION_MODEL_ID
    == DIFFUSION_MODEL_ID
)

print("Diffusion model:", DIFFUSION_MODEL_ID)
print("Cache:", DIFFUSION_CACHE_ROOT)
print("Device:", DEVICE)
print("Dtype:", DIFFUSION_DTYPE)
print("Starting model load...")


synchronize_device()
load_start = time.perf_counter()

(
    diffusion_unet,
    diffusion_scheduler,
) = load_diffusion_components(
    model_id=DIFFUSION_MODEL_ID,
    device=DEVICE,
    dtype=DIFFUSION_DTYPE,
    local_files_only=False,
)

synchronize_device()
diffusion_load_time = (
    time.perf_counter() - load_start
)


unet_parameter = next(
    diffusion_unet.parameters()
)

assert unet_parameter.device == DEVICE
assert unet_parameter.dtype == DIFFUSION_DTYPE
assert not diffusion_unet.training
assert all(
    not parameter.requires_grad
    for parameter
    in diffusion_unet.parameters()
)

diffusion_parameter_count = sum(
    parameter.numel()
    for parameter
    in diffusion_unet.parameters()
)

scheduler_train_timesteps = getattr(
    diffusion_scheduler.config,
    "num_train_timesteps",
    None,
)


DIFFUSION_MANIFEST_PATH = (
    DRIVE_RESULTS_DIR
    / "diffusion_components_manifest.json"
)

diffusion_manifest = {
    "model_id": DIFFUSION_MODEL_ID,
    "timestep": DIFFUSION_TIMESTEP,
    "device": str(DEVICE),
    "dtype": str(DIFFUSION_DTYPE),
    "unet_class": (
        diffusion_unet.__class__.__name__
    ),
    "scheduler_class": (
        diffusion_scheduler
        .__class__.__name__
    ),
    "parameter_count": (
        diffusion_parameter_count
    ),
    "scheduler_train_timesteps": (
        scheduler_train_timesteps
    ),
    "load_time_seconds": (
        diffusion_load_time
    ),
    "source_sha256": (
        "75708d53c4bb616de6ccf85392dd583073"
        "68ec73d118bedfadaf017e10ca622a"
    ),
    "package_versions": {
        package_name: (
            importlib.metadata.version(
                package_name
            )
        )
        for package_name in (
            "torch",
            "diffusers",
            "transformers",
            "accelerate",
            "huggingface-hub",
            "safetensors",
        )
    },
}

temporary_manifest_path = (
    DIFFUSION_MANIFEST_PATH.with_name(
        DIFFUSION_MANIFEST_PATH.name
        + ".tmp"
    )
)

temporary_manifest_path.write_text(
    json.dumps(
        diffusion_manifest,
        indent=2,
    ),
    encoding="utf-8",
)

temporary_manifest_path.replace(
    DIFFUSION_MANIFEST_PATH
)

manifest_hash = hashlib.sha256(
    DIFFUSION_MANIFEST_PATH.read_bytes()
).hexdigest()


print("\nUNet:", diffusion_unet.__class__.__name__)
print(
    "Scheduler:",
    diffusion_scheduler.__class__.__name__,
)
print(
    "Parameters:",
    f"{diffusion_parameter_count:,}",
)
print(
    "Scheduler training timesteps:",
    scheduler_train_timesteps,
)
print(
    "Load time:",
    f"{diffusion_load_time:.2f}s",
)
print("Saved manifest:", DIFFUSION_MANIFEST_PATH)
print("Manifest SHA256:", manifest_hash)

print(
    "\nDIFFUSION COMPONENT LOADING PASSED."
)

Diffusion model: google/ddpm-cifar10-32
Cache: /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/huggingface_cache
Device: cuda:0
Dtype: torch.float32
Starting model load...


Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/699 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors: reconstructing file:   0%|          |  0.00B /  143MB            

diffusion_pytorch_model.safetensors: downloading bytes:           |  0.00B            

scheduler_config.json:   0%|          | 0.00/256 [00:00<?, ?B/s]

There are modules in UNet2DModel that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.



UNet: UNet2DModel
Scheduler: DDPMScheduler
Parameters: 35,746,307
Scheduler training timesteps: 1000
Load time: 20.96s
Saved manifest: /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/diffusion_components_manifest.json
Manifest SHA256: 141aa32484fec56d3b5999ef9651c5b85bc6ac10dcd4460ff6313912a665ac18

DIFFUSION COMPONENT LOADING PASSED.


In [25]:
DIFFUSION_SMOKE_INDICES = [
    0,
    1,
]

smoke_originals = torch.stack(
    [
        TEST_DATASET[index][0]
        for index
        in DIFFUSION_SMOKE_INDICES
    ],
    dim=0,
).to(DEVICE)

smoke_labels = torch.tensor(
    [
        TEST_DATASET[index][1]
        for index
        in DIFFUSION_SMOKE_INDICES
    ],
    dtype=torch.long,
    device=DEVICE,
)

clean_model = CLASSIFIERS["none"]
clean_model.eval()


with torch.no_grad():
    smoke_logits_before = clean_model(
        smoke_originals
    )

smoke_predictions_before = (
    smoke_logits_before.argmax(dim=1)
)


diffusion_generator = torch.Generator(
    device=DEVICE,
).manual_seed(SEED)


synchronize_device()
smoke_defense_start = time.perf_counter()

smoke_purified = purify_images(
    images=smoke_originals,
    unet=diffusion_unet,
    scheduler=diffusion_scheduler,
    timestep=DIFFUSION_TIMESTEP,
    batch_size=1,
    generator=diffusion_generator,
)

synchronize_device()
smoke_defense_time = (
    time.perf_counter()
    - smoke_defense_start
)


assert (
    smoke_purified.shape
    == smoke_originals.shape
)

assert (
    smoke_purified.dtype
    == smoke_originals.dtype
)

assert (
    smoke_purified.device
    == smoke_originals.device
)

assert torch.isfinite(
    smoke_purified
).all()

assert (
    smoke_purified.min().item()
    >= 0.0
)

assert (
    smoke_purified.max().item()
    <= 1.0
)


with torch.no_grad():
    smoke_logits_after = clean_model(
        smoke_purified
    )

smoke_predictions_after = (
    smoke_logits_after.argmax(dim=1)
)


smoke_change = (
    smoke_purified
    - smoke_originals
).flatten(1)

smoke_l2 = smoke_change.norm(
    p=2,
    dim=1,
)

smoke_linf = smoke_change.abs().amax(
    dim=1,
)


DIFFUSION_SMOKE_PATH = (
    DRIVE_RESULTS_DIR
    / "diffusion_purification_smoke_t50.pt"
)

diffusion_smoke_artifact = {
    "indices": DIFFUSION_SMOKE_INDICES,
    "originals": (
        smoke_originals.detach().cpu()
    ),
    "purified": (
        smoke_purified.detach().cpu()
    ),
    "labels": smoke_labels.detach().cpu(),
    "logits_before": (
        smoke_logits_before.detach().cpu()
    ),
    "logits_after": (
        smoke_logits_after.detach().cpu()
    ),
    "predictions_before": (
        smoke_predictions_before
        .detach()
        .cpu()
    ),
    "predictions_after": (
        smoke_predictions_after
        .detach()
        .cpu()
    ),
    "l2_change": smoke_l2.detach().cpu(),
    "linf_change": (
        smoke_linf.detach().cpu()
    ),
    "timestep": DIFFUSION_TIMESTEP,
    "seed": SEED,
    "defense_time_seconds": (
        smoke_defense_time
    ),
    "model_id": DIFFUSION_MODEL_ID,
}

temporary_smoke_path = (
    DIFFUSION_SMOKE_PATH.with_name(
        DIFFUSION_SMOKE_PATH.name + ".tmp"
    )
)

torch.save(
    diffusion_smoke_artifact,
    temporary_smoke_path,
)

temporary_smoke_path.replace(
    DIFFUSION_SMOKE_PATH
)


verified_smoke = torch.load(
    DIFFUSION_SMOKE_PATH,
    map_location="cpu",
    weights_only=False,
)

assert verified_smoke[
    "originals"
].shape == (2, 3, 32, 32)

assert verified_smoke[
    "purified"
].shape == (2, 3, 32, 32)

smoke_hash = hashlib.sha256(
    DIFFUSION_SMOKE_PATH.read_bytes()
).hexdigest()


print("Labels:", smoke_labels.tolist())
print(
    "Predictions before:",
    smoke_predictions_before.tolist(),
)
print(
    "Predictions after:",
    smoke_predictions_after.tolist(),
)
print(
    "L2 changes:",
    [
        round(value, 6)
        for value in smoke_l2.tolist()
    ],
)
print(
    "Linf changes:",
    [
        round(value, 6)
        for value in smoke_linf.tolist()
    ],
)
print(
    "Purification time:",
    f"{smoke_defense_time:.2f}s",
)
print(
    "Purified range:",
    (
        float(smoke_purified.min()),
        float(smoke_purified.max()),
    ),
)
print("Saved:", DIFFUSION_SMOKE_PATH)
print("Smoke SHA256:", smoke_hash)

print(
    "\nREAL DIFFUSION PURIFICATION "
    "SMOKE TEST PASSED."
)

Labels: [3, 8]
Predictions before: [3, 8]
Predictions after: [3, 8]
L2 changes: [2.80239, 2.340241]
Linf changes: [0.194448, 0.202832]
Purification time: 2.00s
Purified range: (0.0, 0.9989732503890991)
Saved: /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/diffusion_purification_smoke_t50.pt
Smoke SHA256: c846ed1be5ddb27e214fd806ec1999f6e24e4755aac1a03c027db8fd59e85b59

REAL DIFFUSION PURIFICATION SMOKE TEST PASSED.


In [26]:
import json


DIFFUSION_SUBSET_SIZE = 200

DIFFUSION_SUBSET_PATH = (
    DRIVE_RESULTS_DIR
    / "diffusion_subset_seed42_n200.pt"
)

DIFFUSION_SUBSET_INDICES_PATH = (
    DRIVE_RESULTS_DIR
    / "diffusion_subset_indices_seed42_n200.json"
)


subset_generator = (
    torch.Generator()
    .manual_seed(SEED)
)

expected_diffusion_indices = (
    torch.randperm(
        len(TEST_DATASET),
        generator=subset_generator,
    )[:DIFFUSION_SUBSET_SIZE]
    .tolist()
)


def validate_diffusion_subset(
    payload,
):
    required_keys = {
        "indices",
        "images",
        "labels",
        "seed",
        "dataset",
        "split",
        "selection_rule",
    }

    assert required_keys.issubset(
        payload
    )

    assert payload["indices"].tolist() == (
        expected_diffusion_indices
    )

    assert payload["images"].shape == (
        DIFFUSION_SUBSET_SIZE,
        3,
        32,
        32,
    )

    assert payload["labels"].shape == (
        DIFFUSION_SUBSET_SIZE,
    )

    assert (
        payload["images"].dtype
        == torch.float32
    )

    assert (
        payload["labels"].dtype
        == torch.long
    )

    assert payload["seed"] == SEED
    assert payload["dataset"] == "CIFAR-10"
    assert payload["split"] == (
        "diffusion_subset"
    )

    assert torch.isfinite(
        payload["images"]
    ).all()

    assert (
        payload["images"].min().item()
        >= 0.0
    )

    assert (
        payload["images"].max().item()
        <= 1.0
    )


if DIFFUSION_SUBSET_PATH.is_file():
    diffusion_subset_payload = torch.load(
        DIFFUSION_SUBSET_PATH,
        map_location="cpu",
        weights_only=False,
    )

    validate_diffusion_subset(
        diffusion_subset_payload
    )

    print(
        "Valid existing diffusion subset loaded."
    )

else:
    diffusion_subset_images = torch.stack(
        [
            TEST_DATASET[index][0]
            for index
            in expected_diffusion_indices
        ],
        dim=0,
    )

    diffusion_subset_labels = torch.tensor(
        [
            TEST_DATASET[index][1]
            for index
            in expected_diffusion_indices
        ],
        dtype=torch.long,
    )

    diffusion_subset_payload = {
        "indices": torch.tensor(
            expected_diffusion_indices,
            dtype=torch.long,
        ),
        "images": diffusion_subset_images,
        "labels": diffusion_subset_labels,
        "seed": SEED,
        "dataset": "CIFAR-10",
        "split": "diffusion_subset",
        "selection_rule": (
            "First 200 indices from "
            "torch.randperm(10000) with "
            "torch.Generator().manual_seed(42)."
        ),
    }

    validate_diffusion_subset(
        diffusion_subset_payload
    )

    temporary_subset_path = (
        DIFFUSION_SUBSET_PATH.with_name(
            DIFFUSION_SUBSET_PATH.name
            + ".tmp"
        )
    )

    torch.save(
        diffusion_subset_payload,
        temporary_subset_path,
    )

    temporary_subset_path.replace(
        DIFFUSION_SUBSET_PATH
    )


indices_metadata = {
    "dataset": "CIFAR-10",
    "source_split": "test",
    "evaluation_split": (
        "diffusion_subset"
    ),
    "num_samples": (
        DIFFUSION_SUBSET_SIZE
    ),
    "seed": SEED,
    "selection_rule": (
        "First 200 indices from "
        "torch.randperm(10000) using "
        "torch.Generator().manual_seed(42)."
    ),
    "indices": expected_diffusion_indices,
}

temporary_indices_path = (
    DIFFUSION_SUBSET_INDICES_PATH
    .with_name(
        DIFFUSION_SUBSET_INDICES_PATH.name
        + ".tmp"
    )
)

temporary_indices_path.write_text(
    json.dumps(
        indices_metadata,
        indent=2,
    ),
    encoding="utf-8",
)

temporary_indices_path.replace(
    DIFFUSION_SUBSET_INDICES_PATH
)


validate_diffusion_subset(
    diffusion_subset_payload
)

subset_file_hash = hashlib.sha256(
    DIFFUSION_SUBSET_PATH.read_bytes()
).hexdigest()

indices_file_hash = hashlib.sha256(
    DIFFUSION_SUBSET_INDICES_PATH
    .read_bytes()
).hexdigest()

label_counts = torch.bincount(
    diffusion_subset_payload["labels"],
    minlength=10,
)


print("Subset size:", DIFFUSION_SUBSET_SIZE)
print(
    "First 20 indices:",
    expected_diffusion_indices[:20],
)
print(
    "Class counts:",
    label_counts.tolist(),
)
print("Saved subset:", DIFFUSION_SUBSET_PATH)
print(
    "Saved indices:",
    DIFFUSION_SUBSET_INDICES_PATH,
)
print("Subset SHA256:", subset_file_hash)
print(
    "Indices SHA256:",
    indices_file_hash,
)

print(
    "\nFIXED DIFFUSION SUBSET PASSED."
)

Subset size: 200
First 20 indices: [7542, 8214, 3698, 2841, 8086, 2550, 2282, 7599, 1462, 2026, 2048, 1871, 6934, 6493, 3819, 717, 6771, 3217, 2519, 6359]
Class counts: [19, 24, 18, 19, 24, 17, 15, 20, 25, 19]
Saved subset: /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/diffusion_subset_seed42_n200.pt
Saved indices: /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/diffusion_subset_indices_seed42_n200.json
Subset SHA256: 363d668e6a4e097702fed9cd04e5cf34bd4c355d8296befbeb8267f4606cf630
Indices SHA256: 02e5c629d12c76ea96bed68625a07da47195217d340a3cf6a2a0924e648303b9

FIXED DIFFUSION SUBSET PASSED.


In [27]:
import gc


DIFFUSION_BATCH_CANDIDATES = (
    1,
    4,
    8,
    16,
)

calibration_images = (
    diffusion_subset_payload[
        "images"
    ][:16].to(DEVICE)
)

calibration_labels = (
    diffusion_subset_payload[
        "labels"
    ][:16].to(DEVICE)
)

diffusion_batch_rows = []


for candidate_batch_size in (
    DIFFUSION_BATCH_CANDIDATES
):
    print("=" * 70)
    print(
        "Testing purification batch size:",
        candidate_batch_size,
    )

    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(
            DEVICE
        )

    candidate_generator = (
        torch.Generator(
            device=DEVICE
        ).manual_seed(SEED)
    )

    try:
        synchronize_device()
        candidate_start = (
            time.perf_counter()
        )

        candidate_purified = purify_images(
            images=calibration_images,
            unet=diffusion_unet,
            scheduler=diffusion_scheduler,
            timestep=DIFFUSION_TIMESTEP,
            batch_size=(
                candidate_batch_size
            ),
            generator=candidate_generator,
        )

        synchronize_device()
        candidate_time = (
            time.perf_counter()
            - candidate_start
        )

        assert (
            candidate_purified.shape
            == calibration_images.shape
        )

        assert (
            candidate_purified.dtype
            == calibration_images.dtype
        )

        assert (
            candidate_purified.device
            == calibration_images.device
        )

        assert torch.isfinite(
            candidate_purified
        ).all()

        assert (
            candidate_purified.min().item()
            >= 0.0
        )

        assert (
            candidate_purified.max().item()
            <= 1.0
        )

        with torch.no_grad():
            candidate_logits = clean_model(
                candidate_purified
            )

        candidate_predictions = (
            candidate_logits.argmax(dim=1)
        )

        candidate_accuracy = (
            candidate_predictions
            .eq(calibration_labels)
            .float()
            .mean()
            .item()
        )

        if DEVICE.type == "cuda":
            peak_memory_mb = (
                torch.cuda
                .max_memory_allocated(
                    DEVICE
                )
                / (1024 ** 2)
            )
        else:
            peak_memory_mb = np.nan

        diffusion_batch_rows.append(
            {
                "batch_size": (
                    candidate_batch_size
                ),
                "num_images": 16,
                "status": "PASSED",
                "total_seconds": (
                    candidate_time
                ),
                "seconds_per_image": (
                    candidate_time / 16
                ),
                "peak_memory_mb": (
                    peak_memory_mb
                ),
                "purified_accuracy": (
                    candidate_accuracy
                ),
                "output_min": float(
                    candidate_purified.min()
                ),
                "output_max": float(
                    candidate_purified.max()
                ),
            }
        )

        print(
            "Time:",
            f"{candidate_time:.2f}s",
        )
        print(
            "Seconds/image:",
            f"{candidate_time / 16:.4f}",
        )
        print(
            "Peak GPU memory:",
            f"{peak_memory_mb:.2f} MB",
        )
        print(
            "Purified accuracy:",
            f"{100 * candidate_accuracy:.2f}%",
        )

        del candidate_purified
        del candidate_logits

    except RuntimeError as error:
        if (
            "out of memory"
            not in str(error).lower()
        ):
            raise

        print(
            "CUDA OUT OF MEMORY at batch size",
            candidate_batch_size,
        )

        diffusion_batch_rows.append(
            {
                "batch_size": (
                    candidate_batch_size
                ),
                "num_images": 16,
                "status": "CUDA_OOM",
                "total_seconds": np.nan,
                "seconds_per_image": np.nan,
                "peak_memory_mb": np.nan,
                "purified_accuracy": np.nan,
                "output_min": np.nan,
                "output_max": np.nan,
            }
        )

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()


diffusion_batch_df = pd.DataFrame(
    diffusion_batch_rows
)

DIFFUSION_BATCH_CALIBRATION_PATH = (
    DRIVE_RESULTS_DIR
    / "diffusion_batch_calibration_t50.csv"
)

temporary_calibration_path = (
    DIFFUSION_BATCH_CALIBRATION_PATH
    .with_name(
        DIFFUSION_BATCH_CALIBRATION_PATH.name
        + ".tmp"
    )
)

diffusion_batch_df.to_csv(
    temporary_calibration_path,
    index=False,
)

temporary_calibration_path.replace(
    DIFFUSION_BATCH_CALIBRATION_PATH
)


successful_batch_rows = (
    diffusion_batch_df.loc[
        diffusion_batch_df[
            "status"
        ].eq("PASSED")
    ]
)

assert not successful_batch_rows.empty

recommended_row = (
    successful_batch_rows
    .sort_values(
        "seconds_per_image"
    )
    .iloc[0]
)

recommended_diffusion_batch_size = int(
    recommended_row["batch_size"]
)

calibration_hash = hashlib.sha256(
    DIFFUSION_BATCH_CALIBRATION_PATH
    .read_bytes()
).hexdigest()


display(diffusion_batch_df)

print(
    "\nRecommended batch size:",
    recommended_diffusion_batch_size,
)
print(
    "Estimated clean 200-image time:",
    (
        f"{200 * recommended_row['seconds_per_image']:.2f}s"
    ),
)
print(
    "Saved:",
    DIFFUSION_BATCH_CALIBRATION_PATH,
)
print(
    "Calibration SHA256:",
    calibration_hash,
)

print(
    "\nDIFFUSION BATCH CALIBRATION PASSED."
)

Testing purification batch size: 1
Time: 15.05s
Seconds/image: 0.9404
Peak GPU memory: 176.94 MB
Purified accuracy: 93.75%
Testing purification batch size: 4
Time: 4.84s
Seconds/image: 0.3022
Peak GPU memory: 198.96 MB
Purified accuracy: 81.25%
Testing purification batch size: 8
Time: 3.64s
Seconds/image: 0.2273
Peak GPU memory: 236.16 MB
Purified accuracy: 93.75%
Testing purification batch size: 16
Time: 2.81s
Seconds/image: 0.1754
Peak GPU memory: 337.05 MB
Purified accuracy: 87.50%


,batch_size,num_images,status,total_seconds,seconds_per_image,peak_memory_mb,purified_accuracy,output_min,output_max
0,1,16,PASSED,15.045639,0.940352,176.939941,0.9375,0.000000,1.0
1,4,16,PASSED,4.835886,0.302243,198.963867,0.8125,0.000000,1.0
2,8,16,PASSED,3.636875,0.227305,236.159668,0.9375,0.003507,1.0
3,16,16,PASSED,2.805740,0.175359,337.054199,0.8750,0.000000,1.0



Recommended batch size: 16
Estimated clean 200-image time: 35.07s
Saved: /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/diffusion_batch_calibration_t50.csv
Calibration SHA256: aa9f36bba6fbb48fdf703fac99b3101ac8030957ea72fcce9f3c1c1b19ffde0e

DIFFUSION BATCH CALIBRATION PASSED.


In [28]:
DIFFUSION_FINAL_BATCH_SIZE = 16

DIFFUSION_CLEAN_CACHE_PATH = (
    DRIVE_RESULTS_DIR
    / (
        "diff_purify_clean_cache_"
        "t50_bs16_seed42_n200.pt"
    )
)

DIFFUSION_EVALUATION_CONFIG_PATH = (
    DRIVE_RESULTS_DIR
    / "diffusion_evaluation_config_t50_seed42.json"
)


def validate_diffusion_clean_cache(
    cache,
):
    required_keys = {
        "indices",
        "original_images",
        "purified_images",
        "labels",
        "raw_logits",
        "purified_logits",
        "raw_predictions",
        "purified_predictions",
        "raw_clean_accuracy",
        "purified_clean_accuracy",
        "prediction_change_count",
        "clean_l2_change",
        "clean_linf_change",
        "defense_time_seconds",
        "classifier_time_seconds",
        "total_time_seconds",
        "model_id",
        "timestep",
        "batch_size",
        "seed",
        "split",
    }

    assert required_keys.issubset(cache)

    assert cache["indices"].tolist() == (
        expected_diffusion_indices
    )

    assert cache["original_images"].shape == (
        200,
        3,
        32,
        32,
    )

    assert cache["purified_images"].shape == (
        200,
        3,
        32,
        32,
    )

    assert cache["labels"].shape == (200,)
    assert cache["raw_logits"].shape == (
        200,
        10,
    )
    assert cache["purified_logits"].shape == (
        200,
        10,
    )
    assert cache["raw_predictions"].shape == (
        200,
    )
    assert cache[
        "purified_predictions"
    ].shape == (200,)

    assert (
        cache["original_images"].dtype
        == torch.float32
    )

    assert (
        cache["purified_images"].dtype
        == torch.float32
    )

    assert torch.isfinite(
        cache["purified_images"]
    ).all()

    assert (
        cache["purified_images"].min().item()
        >= 0.0
    )

    assert (
        cache["purified_images"].max().item()
        <= 1.0
    )

    calculated_raw_accuracy = (
        cache["raw_predictions"]
        .eq(cache["labels"])
        .float()
        .mean()
        .item()
    )

    calculated_purified_accuracy = (
        cache["purified_predictions"]
        .eq(cache["labels"])
        .float()
        .mean()
        .item()
    )

    assert abs(
        calculated_raw_accuracy
        - cache["raw_clean_accuracy"]
    ) < 1e-10

    assert abs(
        calculated_purified_accuracy
        - cache["purified_clean_accuracy"]
    ) < 1e-10

    assert cache["model_id"] == (
        DIFFUSION_MODEL_ID
    )

    assert cache["timestep"] == 50
    assert cache["batch_size"] == 16
    assert cache["seed"] == 42
    assert cache["split"] == (
        "diffusion_subset"
    )


diffusion_clean_cache = None


if DIFFUSION_CLEAN_CACHE_PATH.is_file():
    try:
        loaded_clean_cache = torch.load(
            DIFFUSION_CLEAN_CACHE_PATH,
            map_location="cpu",
            weights_only=False,
        )

        validate_diffusion_clean_cache(
            loaded_clean_cache
        )

        diffusion_clean_cache = (
            loaded_clean_cache
        )

        print(
            "Valid existing diffusion "
            "clean cache loaded."
        )

    except Exception as error:
        print(
            "Existing clean cache invalid:",
            repr(error),
        )
        print(
            "Rebuilding diffusion "
            "clean cache."
        )


if diffusion_clean_cache is None:
    print(
        "Building final clean diffusion cache..."
    )
    print("Samples:", DIFFUSION_SUBSET_SIZE)
    print(
        "Batch size:",
        DIFFUSION_FINAL_BATCH_SIZE,
    )

    subset_images_device = (
        diffusion_subset_payload[
            "images"
        ].to(DEVICE)
    )

    subset_labels_device = (
        diffusion_subset_payload[
            "labels"
        ].to(DEVICE)
    )

    synchronize_device()
    raw_classifier_start = (
        time.perf_counter()
    )

    with torch.no_grad():
        raw_logits = clean_model(
            subset_images_device
        )

    synchronize_device()
    raw_classifier_time = (
        time.perf_counter()
        - raw_classifier_start
    )

    raw_predictions = raw_logits.argmax(
        dim=1
    )

    final_clean_generator = (
        torch.Generator(
            device=DEVICE
        ).manual_seed(SEED)
    )

    synchronize_device()
    defense_start = time.perf_counter()

    purified_clean_images = purify_images(
        images=subset_images_device,
        unet=diffusion_unet,
        scheduler=diffusion_scheduler,
        timestep=DIFFUSION_TIMESTEP,
        batch_size=(
            DIFFUSION_FINAL_BATCH_SIZE
        ),
        generator=final_clean_generator,
    )

    synchronize_device()
    clean_defense_time = (
        time.perf_counter()
        - defense_start
    )

    synchronize_device()
    purified_classifier_start = (
        time.perf_counter()
    )

    with torch.no_grad():
        purified_logits = clean_model(
            purified_clean_images
        )

    synchronize_device()
    purified_classifier_time = (
        time.perf_counter()
        - purified_classifier_start
    )

    purified_predictions = (
        purified_logits.argmax(dim=1)
    )

    clean_change = (
        purified_clean_images
        - subset_images_device
    ).flatten(1)

    clean_l2_change = clean_change.norm(
        p=2,
        dim=1,
    )

    clean_linf_change = (
        clean_change.abs().amax(dim=1)
    )

    raw_clean_accuracy = (
        raw_predictions
        .eq(subset_labels_device)
        .float()
        .mean()
        .item()
    )

    purified_clean_accuracy = (
        purified_predictions
        .eq(subset_labels_device)
        .float()
        .mean()
        .item()
    )

    prediction_change_count = int(
        raw_predictions.ne(
            purified_predictions
        ).sum().item()
    )

    classifier_time = (
        raw_classifier_time
        + purified_classifier_time
    )

    clean_total_time = (
        clean_defense_time
        + classifier_time
    )

    diffusion_clean_cache = {
        "indices": (
            diffusion_subset_payload[
                "indices"
            ].clone()
        ),
        "original_images": (
            subset_images_device
            .detach()
            .cpu()
        ),
        "purified_images": (
            purified_clean_images
            .detach()
            .cpu()
        ),
        "labels": (
            subset_labels_device
            .detach()
            .cpu()
        ),
        "raw_logits": (
            raw_logits.detach().cpu()
        ),
        "purified_logits": (
            purified_logits
            .detach()
            .cpu()
        ),
        "raw_predictions": (
            raw_predictions
            .detach()
            .cpu()
        ),
        "purified_predictions": (
            purified_predictions
            .detach()
            .cpu()
        ),
        "raw_clean_accuracy": (
            raw_clean_accuracy
        ),
        "purified_clean_accuracy": (
            purified_clean_accuracy
        ),
        "prediction_change_count": (
            prediction_change_count
        ),
        "clean_l2_change": (
            clean_l2_change
            .detach()
            .cpu()
        ),
        "clean_linf_change": (
            clean_linf_change
            .detach()
            .cpu()
        ),
        "defense_time_seconds": (
            clean_defense_time
        ),
        "classifier_time_seconds": (
            classifier_time
        ),
        "total_time_seconds": (
            clean_total_time
        ),
        "model_id": (
            DIFFUSION_MODEL_ID
        ),
        "timestep": (
            DIFFUSION_TIMESTEP
        ),
        "batch_size": (
            DIFFUSION_FINAL_BATCH_SIZE
        ),
        "seed": SEED,
        "split": "diffusion_subset",
    }

    validate_diffusion_clean_cache(
        diffusion_clean_cache
    )

    temporary_clean_cache_path = (
        DIFFUSION_CLEAN_CACHE_PATH
        .with_name(
            DIFFUSION_CLEAN_CACHE_PATH.name
            + ".tmp"
        )
    )

    torch.save(
        diffusion_clean_cache,
        temporary_clean_cache_path,
    )

    temporary_clean_cache_path.replace(
        DIFFUSION_CLEAN_CACHE_PATH
    )


validate_diffusion_clean_cache(
    diffusion_clean_cache
)


subset_hash = hashlib.sha256(
    DIFFUSION_SUBSET_PATH.read_bytes()
).hexdigest()

indices_hash = hashlib.sha256(
    DIFFUSION_SUBSET_INDICES_PATH
    .read_bytes()
).hexdigest()

batch_calibration_hash = hashlib.sha256(
    DIFFUSION_BATCH_CALIBRATION_PATH
    .read_bytes()
).hexdigest()

clean_cache_hash = hashlib.sha256(
    DIFFUSION_CLEAN_CACHE_PATH
    .read_bytes()
).hexdigest()


diffusion_evaluation_config = {
    "defense_id": "diff_purify",
    "diffusion_model_id": (
        DIFFUSION_MODEL_ID
    ),
    "classifier_checkpoint": (
        CHECKPOINT_NAMES["none"]
    ),
    "timestep": (
        DIFFUSION_TIMESTEP
    ),
    "purification_batch_size": (
        DIFFUSION_FINAL_BATCH_SIZE
    ),
    "num_samples": (
        DIFFUSION_SUBSET_SIZE
    ),
    "split": "diffusion_subset",
    "seed": SEED,
    "selection_rule": (
        diffusion_subset_payload[
            "selection_rule"
        ]
    ),
    "source_sha256": (
        "75708d53c4bb616de6ccf85392dd583073"
        "68ec73d118bedfadaf017e10ca622a"
    ),
    "subset_sha256": subset_hash,
    "indices_sha256": indices_hash,
    "batch_calibration_sha256": (
        batch_calibration_hash
    ),
    "clean_cache_sha256": (
        clean_cache_hash
    ),
}

temporary_config_path = (
    DIFFUSION_EVALUATION_CONFIG_PATH
    .with_name(
        DIFFUSION_EVALUATION_CONFIG_PATH.name
        + ".tmp"
    )
)

temporary_config_path.write_text(
    json.dumps(
        diffusion_evaluation_config,
        indent=2,
    ),
    encoding="utf-8",
)

temporary_config_path.replace(
    DIFFUSION_EVALUATION_CONFIG_PATH
)

evaluation_config_hash = hashlib.sha256(
    DIFFUSION_EVALUATION_CONFIG_PATH
    .read_bytes()
).hexdigest()


print(
    "\nRaw subset clean accuracy:",
    (
        f"{100 * diffusion_clean_cache['raw_clean_accuracy']:.2f}%"
    ),
)
print(
    "After-purification clean accuracy:",
    (
        f"{100 * diffusion_clean_cache['purified_clean_accuracy']:.2f}%"
    ),
)
print(
    "Accuracy change:",
    (
        f"{100 * (
            diffusion_clean_cache[
                'purified_clean_accuracy'
            ]
            - diffusion_clean_cache[
                'raw_clean_accuracy'
            ]
        ):+.2f} percentage points"
    ),
)
print(
    "Changed predictions:",
    diffusion_clean_cache[
        "prediction_change_count"
    ],
    "/",
    DIFFUSION_SUBSET_SIZE,
)
print(
    "Mean clean L2 change:",
    (
        f"{diffusion_clean_cache['clean_l2_change'].mean().item():.6f}"
    ),
)
print(
    "Mean clean Linf change:",
    (
        f"{diffusion_clean_cache['clean_linf_change'].mean().item():.6f}"
    ),
)
print(
    "Defense time:",
    (
        f"{diffusion_clean_cache['defense_time_seconds']:.2f}s"
    ),
)
print(
    "Classifier time:",
    (
        f"{diffusion_clean_cache['classifier_time_seconds']:.2f}s"
    ),
)
print("Saved cache:", DIFFUSION_CLEAN_CACHE_PATH)
print("Clean cache SHA256:", clean_cache_hash)
print(
    "Saved config:",
    DIFFUSION_EVALUATION_CONFIG_PATH,
)
print(
    "Config SHA256:",
    evaluation_config_hash,
)

print(
    "\nFINAL CLEAN DIFFUSION CACHE PASSED."
)

Building final clean diffusion cache...
Samples: 200
Batch size: 16

Raw subset clean accuracy: 87.50%
After-purification clean accuracy: 82.00%
Accuracy change: -5.50 percentage points
Changed predictions: 30 / 200
Mean clean L2 change: 2.423623
Mean clean Linf change: 0.207885
Defense time: 35.79s
Classifier time: 0.02s
Saved cache: /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/diff_purify_clean_cache_t50_bs16_seed42_n200.pt
Clean cache SHA256: ff3dd451999fce8e971eeba23d4024ee82e0ee671a998f1ea1941159e6d3fe16
Saved config: /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/diffusion_evaluation_config_t50_seed42.json
Config SHA256: 157a0b3f94360c1e1710074b29cb6bbb5ff27afcb0aa749f9a8218d9243744e2

FINAL CLEAN DIFFUSION CACHE PASSED.


In [29]:
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset

import gc
import hashlib
import json
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from src.attacks.fgsm import fgsm_attack
from src.attacks.pgd import pgd_attack
from src.attacks.deepfool import deepfool_attack
from src.attacks.cw_l2 import cw_l2_attack

from src.defenses.diffusion_purification import (
    purify_images,
)


# --------------------------------------------------
# مسیرهای دائمی
# --------------------------------------------------

DIFFUSION_RESULTS_DIR = Path(
    "/content/drive/MyDrive/"
    "AI-Project-Adversarial-Attack-Defense/results"
)

DIFFUSION_CLEAN_CACHE_PATH = (
    DIFFUSION_RESULTS_DIR
    / "diff_purify_clean_cache_t50_bs16_seed42_n200.pt"
)

DIFFUSION_SUBSET_INDICES_PATH = (
    DIFFUSION_RESULTS_DIR
    / "diffusion_subset_indices_seed42_n200.json"
)

DIFFUSION_FINAL_METRICS_PATH = (
    DIFFUSION_RESULTS_DIR
    / "evaluate_colab_diff_purify_final.csv"
)

DIFFUSION_FINAL_EXAMPLES_DIR = (
    DIFFUSION_RESULTS_DIR
    / "final_attack_examples"
)

DIFFUSION_FINAL_EXAMPLES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------
# قرارداد ثابت
# --------------------------------------------------

CANONICAL_METRIC_COLUMNS = [
    "run_id",
    "model_id",
    "defense_id",
    "attack_id",
    "split",
    "num_samples",
    "seed",
    "epsilon",
    "alpha",
    "attack_steps",
    "clean_accuracy",
    "robust_accuracy",
    "attack_success_rate",
    "mean_l2",
    "mean_l2_successful",
    "mean_linf",
    "mean_linf_successful",
    "attack_time_seconds",
    "defense_time_seconds",
    "total_time_seconds",
    "checkpoint_name",
]

DIFFUSION_SEED = 42
DIFFUSION_TIMESTEP = 50
DIFFUSION_BATCH_SIZE = 16
DIFFUSION_NUM_SAMPLES = 200


# --------------------------------------------------
# یافتن UNet و Scheduler لودشده
# --------------------------------------------------

DIFFUSION_UNET = None

for variable_name in (
    "diffusion_unet",
    "DIFFUSION_UNET",
    "unet",
):
    candidate = globals().get(variable_name)

    if (
        isinstance(candidate, nn.Module)
        and candidate.__class__.__name__
        == "UNet2DModel"
    ):
        DIFFUSION_UNET = candidate
        break

if DIFFUSION_UNET is None:
    for candidate in list(globals().values()):
        if (
            isinstance(candidate, nn.Module)
            and candidate.__class__.__name__
            == "UNet2DModel"
        ):
            DIFFUSION_UNET = candidate
            break


DIFFUSION_SCHEDULER = None

for variable_name in (
    "diffusion_scheduler",
    "DIFFUSION_SCHEDULER",
    "scheduler",
):
    candidate = globals().get(variable_name)

    if (
        candidate is not None
        and candidate.__class__.__name__
        == "DDPMScheduler"
    ):
        DIFFUSION_SCHEDULER = candidate
        break

if DIFFUSION_SCHEDULER is None:
    for candidate in list(globals().values()):
        if (
            candidate is not None
            and candidate.__class__.__name__
            == "DDPMScheduler"
        ):
            DIFFUSION_SCHEDULER = candidate
            break

assert DIFFUSION_UNET is not None
assert DIFFUSION_SCHEDULER is not None


# --------------------------------------------------
# مدل classifier پایه
# --------------------------------------------------

assert "CLASSIFIERS" in globals()
assert "none" in CLASSIFIERS

DIFFUSION_CLASSIFIER = CLASSIFIERS["none"]
DIFFUSION_CLASSIFIER.eval()

DIFFUSION_DEVICE = next(
    DIFFUSION_CLASSIFIER.parameters()
).device

assert (
    next(DIFFUSION_UNET.parameters()).device
    == DIFFUSION_DEVICE
)


# --------------------------------------------------
# زیرمجموعه ثابت
# --------------------------------------------------

indices_document = json.loads(
    DIFFUSION_SUBSET_INDICES_PATH.read_text(
        encoding="utf-8"
    )
)

DIFFUSION_SUBSET_INDICES = [
    int(index)
    for index in indices_document["indices"]
]

assert len(DIFFUSION_SUBSET_INDICES) == 200
assert len(set(DIFFUSION_SUBSET_INDICES)) == 200

subset_records = [
    TEST_DATASET[index]
    for index in DIFFUSION_SUBSET_INDICES
]

DIFFUSION_RAW_IMAGES = torch.stack(
    [record[0] for record in subset_records]
).cpu()

DIFFUSION_LABELS = torch.tensor(
    [int(record[1]) for record in subset_records],
    dtype=torch.long,
)

assert DIFFUSION_RAW_IMAGES.shape == (
    200,
    3,
    32,
    32,
)


# --------------------------------------------------
# کش پاکِ purification
# --------------------------------------------------

diffusion_clean_cache = torch.load(
    DIFFUSION_CLEAN_CACHE_PATH,
    map_location="cpu",
    weights_only=False,
)

_MISSING = object()


def diffusion_cache_value(
    *candidate_keys,
    default=_MISSING,
):
    for key in candidate_keys:
        if key in diffusion_clean_cache:
            return diffusion_clean_cache[key]

    if default is not _MISSING:
        return default

    raise KeyError(
        "None of these keys exist in clean cache: "
        f"{candidate_keys}. Available keys: "
        f"{sorted(diffusion_clean_cache.keys())}"
    )


DIFFUSION_PURIFIED_CLEAN_IMAGES = (
    diffusion_cache_value(
        "purified_images",
        "clean_purified_images",
        "purified_clean_images",
    )
    .detach()
    .cpu()
)

cached_labels = diffusion_cache_value(
    "labels",
    "subset_labels",
    default=None,
)

if cached_labels is not None:
    assert torch.equal(
        torch.as_tensor(cached_labels).long().cpu(),
        DIFFUSION_LABELS,
    )

cached_indices = diffusion_cache_value(
    "indices",
    "subset_indices",
    default=None,
)

if cached_indices is not None:
    assert [
        int(value)
        for value in torch.as_tensor(
            cached_indices
        ).tolist()
    ] == DIFFUSION_SUBSET_INDICES

assert (
    DIFFUSION_PURIFIED_CLEAN_IMAGES.shape
    == DIFFUSION_RAW_IMAGES.shape
)

assert (
    DIFFUSION_PURIFIED_CLEAN_IMAGES.min().item()
    >= 0.0
)

assert (
    DIFFUSION_PURIFIED_CLEAN_IMAGES.max().item()
    <= 1.0
)


DIFFUSION_CLEAN_DEFENSE_TIME = float(
    diffusion_cache_value(
        "defense_time_seconds",
        "purification_time_seconds",
        "clean_defense_time_seconds",
    )
)

DIFFUSION_CLEAN_CLASSIFIER_TIME = float(
    diffusion_cache_value(
        "classifier_time_seconds",
        "clean_classifier_time_seconds",
        default=0.0,
    )
)


# --------------------------------------------------
# پیش‌بینی پاکِ دفاع‌شده
# --------------------------------------------------

clean_logit_batches = []

for start in range(
    0,
    DIFFUSION_NUM_SAMPLES,
    128,
):
    clean_batch = (
        DIFFUSION_PURIFIED_CLEAN_IMAGES[
            start:start + 128
        ].to(DIFFUSION_DEVICE)
    )

    with torch.no_grad():
        clean_logit_batches.append(
            DIFFUSION_CLASSIFIER(
                clean_batch
            ).cpu()
        )

DIFFUSION_CLEAN_LOGITS = torch.cat(
    clean_logit_batches,
    dim=0,
)

DIFFUSION_CLEAN_PREDICTIONS = (
    DIFFUSION_CLEAN_LOGITS.argmax(dim=1)
)

DIFFUSION_CLEAN_CORRECT = (
    DIFFUSION_CLEAN_PREDICTIONS.eq(
        DIFFUSION_LABELS
    )
)

DIFFUSION_CLEAN_ACCURACY = float(
    DIFFUSION_CLEAN_CORRECT.float().mean().item()
)

assert abs(
    DIFFUSION_CLEAN_ACCURACY - 0.82
) < 1e-6


# --------------------------------------------------
# هفت پیکربندی نهایی
# --------------------------------------------------

DIFFUSION_ATTACK_CONFIGS = [
    {
        "run_id": (
            "diff_purify__fgsm__eps2__"
            "t50__seed42"
        ),
        "attack_id": "fgsm",
        "epsilon": 2 / 255,
        "alpha": np.nan,
        "attack_steps": 1,
        "attack_batch_size": 64,
    },
    {
        "run_id": (
            "diff_purify__fgsm__eps4__"
            "t50__seed42"
        ),
        "attack_id": "fgsm",
        "epsilon": 4 / 255,
        "alpha": np.nan,
        "attack_steps": 1,
        "attack_batch_size": 64,
    },
    {
        "run_id": (
            "diff_purify__fgsm__eps8__"
            "t50__seed42"
        ),
        "attack_id": "fgsm",
        "epsilon": 8 / 255,
        "alpha": np.nan,
        "attack_steps": 1,
        "attack_batch_size": 64,
    },
    {
        "run_id": (
            "diff_purify__fgsm__eps16__"
            "t50__seed42"
        ),
        "attack_id": "fgsm",
        "epsilon": 16 / 255,
        "alpha": np.nan,
        "attack_steps": 1,
        "attack_batch_size": 64,
    },
    {
        "run_id": (
            "diff_purify__pgd__eps8__"
            "alpha2__steps20__t50__seed42"
        ),
        "attack_id": "pgd",
        "epsilon": 8 / 255,
        "alpha": 2 / 255,
        "attack_steps": 20,
        "attack_batch_size": 64,
    },
    {
        "run_id": (
            "diff_purify__deepfool__"
            "steps50__t50__seed42"
        ),
        "attack_id": "deepfool",
        "epsilon": np.nan,
        "alpha": np.nan,
        "attack_steps": 50,
        "attack_batch_size": 8,
    },
    {
        "run_id": (
            "diff_purify__cw_l2__"
            "steps100__t50__seed42"
        ),
        "attack_id": "cw_l2",
        "epsilon": np.nan,
        "alpha": np.nan,
        "attack_steps": 100,
        "attack_batch_size": 32,
    },
]

assert len(DIFFUSION_ATTACK_CONFIGS) == 7

print("Device:", DIFFUSION_DEVICE)
print("Subset samples:", DIFFUSION_NUM_SAMPLES)
print("Defended clean accuracy:", DIFFUSION_CLEAN_ACCURACY)
print("Clean defense time:", DIFFUSION_CLEAN_DEFENSE_TIME)
print("Attack configurations:", len(DIFFUSION_ATTACK_CONFIGS))
print("Metrics:", DIFFUSION_FINAL_METRICS_PATH)

print("\nDIFFUSION ATTACK EVALUATION SETUP PASSED.")

Device: cuda:0
Subset samples: 200
Defended clean accuracy: 0.8199999928474426
Clean defense time: 35.79009349300031
Attack configurations: 7
Metrics: /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/evaluate_colab_diff_purify_final.csv

DIFFUSION ATTACK EVALUATION SETUP PASSED.


In [30]:
def synchronize_diffusion_device():
    if DIFFUSION_DEVICE.type == "cuda":
        torch.cuda.synchronize(
            DIFFUSION_DEVICE
        )


def generate_diffusion_raw_adversarials(
    attack_config,
):
    attack_id = attack_config["attack_id"]

    loader = DataLoader(
        TensorDataset(
            DIFFUSION_RAW_IMAGES,
            DIFFUSION_LABELS,
        ),
        batch_size=attack_config[
            "attack_batch_size"
        ],
        shuffle=False,
        num_workers=0,
        pin_memory=(
            DIFFUSION_DEVICE.type == "cuda"
        ),
    )

    adversarial_batches = []

    cpu_rng_state = torch.random.get_rng_state()

    cuda_rng_states = (
        torch.cuda.get_rng_state_all()
        if DIFFUSION_DEVICE.type == "cuda"
        else None
    )

    torch.manual_seed(DIFFUSION_SEED)

    if DIFFUSION_DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(
            DIFFUSION_SEED
        )

    try:
        synchronize_diffusion_device()
        attack_start = time.perf_counter()

        for images, labels in loader:
            images = images.to(
                DIFFUSION_DEVICE,
                non_blocking=True,
            )

            labels = labels.to(
                DIFFUSION_DEVICE,
                non_blocking=True,
            )

            if attack_id == "fgsm":
                adversarial_images = fgsm_attack(
                    model=DIFFUSION_CLASSIFIER,
                    images=images,
                    labels=labels,
                    epsilon=attack_config[
                        "epsilon"
                    ],
                )

            elif attack_id == "pgd":
                adversarial_images = pgd_attack(
                    model=DIFFUSION_CLASSIFIER,
                    images=images,
                    labels=labels,
                    epsilon=attack_config[
                        "epsilon"
                    ],
                    alpha=attack_config[
                        "alpha"
                    ],
                    steps=attack_config[
                        "attack_steps"
                    ],
                    random_start=True,
                    restarts=1,
                )

            elif attack_id == "deepfool":
                adversarial_images = deepfool_attack(
                    model=DIFFUSION_CLASSIFIER,
                    images=images,
                    labels=labels,
                    max_steps=50,
                    overshoot=0.02,
                    num_classes=10,
                )

            elif attack_id == "cw_l2":
                adversarial_images = cw_l2_attack(
                    model=DIFFUSION_CLASSIFIER,
                    images=images,
                    labels=labels,
                    c=1.0,
                    kappa=0.0,
                    learning_rate=0.01,
                    steps=100,
                )

            else:
                raise ValueError(
                    f"Unsupported attack: {attack_id}"
                )

            assert (
                adversarial_images.shape
                == images.shape
            )

            assert (
                adversarial_images.min().item()
                >= 0.0
            )

            assert (
                adversarial_images.max().item()
                <= 1.0
            )

            adversarial_batches.append(
                adversarial_images.detach().cpu()
            )

        synchronize_diffusion_device()

        attack_time = (
            time.perf_counter() - attack_start
        )

    finally:
        torch.random.set_rng_state(
            cpu_rng_state
        )

        if cuda_rng_states is not None:
            torch.cuda.set_rng_state_all(
                cuda_rng_states
            )

    raw_adversarials = torch.cat(
        adversarial_batches,
        dim=0,
    )

    if attack_id in ("fgsm", "pgd"):
        measured_linf = (
            raw_adversarials
            - DIFFUSION_RAW_IMAGES
        ).abs().amax().item()

        assert measured_linf <= (
            attack_config["epsilon"] + 1e-6
        )

    return raw_adversarials, attack_time


def evaluate_diffusion_attack(
    attack_config,
):
    raw_adversarials, attack_time = (
        generate_diffusion_raw_adversarials(
            attack_config
        )
    )

    purification_generator = torch.Generator(
        device=DIFFUSION_DEVICE
    ).manual_seed(DIFFUSION_SEED)

    synchronize_diffusion_device()
    defense_start = time.perf_counter()

    purified_adversarials = purify_images(
        images=raw_adversarials.to(
            DIFFUSION_DEVICE
        ),
        unet=DIFFUSION_UNET,
        scheduler=DIFFUSION_SCHEDULER,
        timestep=DIFFUSION_TIMESTEP,
        batch_size=DIFFUSION_BATCH_SIZE,
        generator=purification_generator,
    )

    synchronize_diffusion_device()

    adversarial_defense_time = (
        time.perf_counter() - defense_start
    )

    adversarial_logit_batches = []

    synchronize_diffusion_device()
    classifier_start = time.perf_counter()

    for start in range(
        0,
        DIFFUSION_NUM_SAMPLES,
        128,
    ):
        with torch.no_grad():
            adversarial_logit_batches.append(
                DIFFUSION_CLASSIFIER(
                    purified_adversarials[
                        start:start + 128
                    ]
                ).cpu()
            )

    synchronize_diffusion_device()

    adversarial_classifier_time = (
        time.perf_counter()
        - classifier_start
    )

    adversarial_logits = torch.cat(
        adversarial_logit_batches,
        dim=0,
    )

    purified_adversarials_cpu = (
        purified_adversarials.detach().cpu()
    )

    adversarial_predictions = (
        adversarial_logits.argmax(dim=1)
    )

    adversarial_correct = (
        adversarial_predictions.eq(
            DIFFUSION_LABELS
        )
    )

    successful = (
        DIFFUSION_CLEAN_CORRECT
        & (~adversarial_correct)
    )

    clean_correct_total = int(
        DIFFUSION_CLEAN_CORRECT.sum().item()
    )

    adversarial_correct_total = int(
        adversarial_correct.sum().item()
    )

    successful_total = int(
        successful.sum().item()
    )

    perturbations = (
        raw_adversarials
        - DIFFUSION_RAW_IMAGES
    ).flatten(1)

    l2_norms = perturbations.norm(
        p=2,
        dim=1,
    )

    linf_norms = perturbations.abs().amax(
        dim=1,
    )

    attack_success_rate = (
        successful_total / clean_correct_total
        if clean_correct_total > 0
        else float("nan")
    )

    mean_l2_successful = (
        float(
            l2_norms[successful].mean().item()
        )
        if successful_total > 0
        else float("nan")
    )

    mean_linf_successful = (
        float(
            linf_norms[successful].mean().item()
        )
        if successful_total > 0
        else float("nan")
    )

    complete_defense_time = (
        DIFFUSION_CLEAN_DEFENSE_TIME
        + adversarial_defense_time
    )

    complete_total_time = (
        attack_time
        + complete_defense_time
        + DIFFUSION_CLEAN_CLASSIFIER_TIME
        + adversarial_classifier_time
    )

    row = {
        "run_id": attack_config["run_id"],
        "model_id": "resnet20",
        "defense_id": "diff_purify",
        "attack_id": attack_config[
            "attack_id"
        ],
        "split": "diffusion_subset",
        "num_samples": DIFFUSION_NUM_SAMPLES,
        "seed": DIFFUSION_SEED,
        "epsilon": attack_config[
            "epsilon"
        ],
        "alpha": attack_config["alpha"],
        "attack_steps": attack_config[
            "attack_steps"
        ],
        "clean_accuracy": (
            clean_correct_total
            / DIFFUSION_NUM_SAMPLES
        ),
        "robust_accuracy": (
            adversarial_correct_total
            / DIFFUSION_NUM_SAMPLES
        ),
        "attack_success_rate": (
            attack_success_rate
        ),
        "mean_l2": float(
            l2_norms.mean().item()
        ),
        "mean_l2_successful": (
            mean_l2_successful
        ),
        "mean_linf": float(
            linf_norms.mean().item()
        ),
        "mean_linf_successful": (
            mean_linf_successful
        ),
        "attack_time_seconds": (
            attack_time
        ),
        "defense_time_seconds": (
            complete_defense_time
        ),
        "total_time_seconds": (
            complete_total_time
        ),
        "checkpoint_name": (
            "resnet20_clean_best.pt"
        ),
    }

    selected = torch.where(
        successful
    )[0][:8]

    examples = {
        "run_id": attack_config["run_id"],
        "subset_indices": torch.tensor(
            DIFFUSION_SUBSET_INDICES
        )[selected],
        "originals": (
            DIFFUSION_RAW_IMAGES[selected]
        ),
        "adversarials": (
            raw_adversarials[selected]
        ),
        "purified_originals": (
            DIFFUSION_PURIFIED_CLEAN_IMAGES[
                selected
            ]
        ),
        "purified_adversarials": (
            purified_adversarials_cpu[
                selected
            ]
        ),
        "labels": DIFFUSION_LABELS[selected],
        "clean_logits": (
            DIFFUSION_CLEAN_LOGITS[selected]
        ),
        "adversarial_logits": (
            adversarial_logits[selected]
        ),
        "successful_samples": (
            successful_total
        ),
        "clean_correct_samples": (
            clean_correct_total
        ),
        "adversarial_correct_samples": (
            adversarial_correct_total
        ),
        "adversarial_defense_time_seconds": (
            adversarial_defense_time
        ),
        "adversarial_classifier_time_seconds": (
            adversarial_classifier_time
        ),
        "evaluation_protocol": (
            "non_adaptive_attack_then_purify"
        ),
    }

    return row, examples


print("DIFFUSION ATTACK EVALUATOR DEFINED.")

DIFFUSION ATTACK EVALUATOR DEFINED.


In [31]:
if DIFFUSION_FINAL_METRICS_PATH.is_file():
    diffusion_final_df = pd.read_csv(
        DIFFUSION_FINAL_METRICS_PATH
    )

    assert list(
        diffusion_final_df.columns
    ) == CANONICAL_METRIC_COLUMNS

    print(
        "Existing diffusion results loaded:",
        len(diffusion_final_df),
        "rows",
    )

else:
    diffusion_final_df = pd.DataFrame(
        columns=CANONICAL_METRIC_COLUMNS
    )

    print(
        "No previous diffusion attack "
        "results found."
    )


def diffusion_examples_path(run_id):
    return (
        DIFFUSION_FINAL_EXAMPLES_DIR
        / f"{run_id}__examples.pt"
    )


def diffusion_run_is_complete(run_id):
    if diffusion_final_df.empty:
        return False

    matching = diffusion_final_df.loc[
        diffusion_final_df["run_id"].eq(
            run_id
        )
        & diffusion_final_df[
            "num_samples"
        ].eq(DIFFUSION_NUM_SAMPLES)
        & diffusion_final_df["split"].eq(
            "diffusion_subset"
        )
    ]

    return (
        len(matching) == 1
        and diffusion_examples_path(
            run_id
        ).is_file()
    )


run_order = {
    config["run_id"]: position
    for position, config
    in enumerate(DIFFUSION_ATTACK_CONFIGS)
}


for attack_config in DIFFUSION_ATTACK_CONFIGS:
    run_id = attack_config["run_id"]

    if diffusion_run_is_complete(run_id):
        print("=" * 70)
        print("SKIPPED — completed:", run_id)
        continue

    print("=" * 70)
    print("FINAL RUN STARTED:", run_id)
    print("Samples:", DIFFUSION_NUM_SAMPLES)

    row, examples = evaluate_diffusion_attack(
        attack_config
    )

    if not diffusion_final_df.empty:
        diffusion_final_df = (
            diffusion_final_df.loc[
                ~diffusion_final_df[
                    "run_id"
                ].eq(run_id)
            ].copy()
        )

    new_row_df = pd.DataFrame(
        [row],
        columns=CANONICAL_METRIC_COLUMNS,
    )

    if diffusion_final_df.empty:
        diffusion_final_df = new_row_df

    else:
        diffusion_final_df = pd.concat(
            [
                diffusion_final_df,
                new_row_df,
            ],
            ignore_index=True,
        )

    diffusion_final_df["_run_order"] = (
        diffusion_final_df["run_id"].map(
            run_order
        )
    )

    diffusion_final_df = (
        diffusion_final_df
        .sort_values("_run_order")
        .drop(columns="_run_order")
        .reset_index(drop=True)
    )

    temporary_metrics_path = (
        DIFFUSION_FINAL_METRICS_PATH
        .with_name(
            DIFFUSION_FINAL_METRICS_PATH.name
            + ".tmp"
        )
    )

    diffusion_final_df.to_csv(
        temporary_metrics_path,
        index=False,
    )

    temporary_metrics_path.replace(
        DIFFUSION_FINAL_METRICS_PATH
    )

    examples_path = diffusion_examples_path(
        run_id
    )

    temporary_examples_path = (
        examples_path.with_name(
            examples_path.name + ".tmp"
        )
    )

    torch.save(
        examples,
        temporary_examples_path,
    )

    temporary_examples_path.replace(
        examples_path
    )

    metrics_hash = hashlib.sha256(
        DIFFUSION_FINAL_METRICS_PATH.read_bytes()
    ).hexdigest()

    print(
        f"Clean accuracy: "
        f"{100 * row['clean_accuracy']:.2f}%"
    )
    print(
        f"Robust accuracy: "
        f"{100 * row['robust_accuracy']:.2f}%"
    )
    print(
        f"Attack success rate: "
        f"{100 * row['attack_success_rate']:.2f}%"
    )
    print(
        f"Mean L2 successful: "
        f"{row['mean_l2_successful']:.6f}"
    )
    print(
        f"Attack time: "
        f"{row['attack_time_seconds']:.2f}s"
    )
    print(
        f"Defense time: "
        f"{row['defense_time_seconds']:.2f}s"
    )
    print(
        "Successful examples saved:",
        examples["labels"].shape[0],
    )
    print("Saved metrics:", DIFFUSION_FINAL_METRICS_PATH)
    print("Saved examples:", examples_path)
    print("Metrics SHA256:", metrics_hash)

    gc.collect()

    if DIFFUSION_DEVICE.type == "cuda":
        torch.cuda.empty_cache()


print("\nCURRENT DIFFUSION RESULTS:")

display(
    diffusion_final_df[
        [
            "attack_id",
            "epsilon",
            "attack_steps",
            "clean_accuracy",
            "robust_accuracy",
            "attack_success_rate",
            "mean_l2_successful",
            "mean_linf_successful",
            "attack_time_seconds",
            "defense_time_seconds",
            "total_time_seconds",
        ]
    ]
)

No previous diffusion attack results found.
FINAL RUN STARTED: diff_purify__fgsm__eps2__t50__seed42
Samples: 200
Clean accuracy: 82.00%
Robust accuracy: 74.50%
Attack success rate: 9.15%
Mean L2 successful: 0.433618
Attack time: 0.05s
Defense time: 72.60s
Successful examples saved: 8
Saved metrics: /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/evaluate_colab_diff_purify_final.csv
Saved examples: /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/final_attack_examples/diff_purify__fgsm__eps2__t50__seed42__examples.pt
Metrics SHA256: 63b52cc86d9600e529a40b339e45af1c98b815b12420fca1b64e4b7cec8b12eb
FINAL RUN STARTED: diff_purify__fgsm__eps4__t50__seed42
Samples: 200
Clean accuracy: 82.00%
Robust accuracy: 67.00%
Attack success rate: 18.29%
Mean L2 successful: 0.860903
Attack time: 0.05s
Defense time: 69.78s
Successful examples saved: 8
Saved metrics: /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/evaluate_colab_diff_purify_fin

,attack_id,epsilon,attack_steps,clean_accuracy,robust_accuracy,attack_success_rate,mean_l2_successful,mean_linf_successful,attack_time_seconds,defense_time_seconds,total_time_seconds
0,fgsm,0.007843,1,0.82,0.745,0.091463,0.433618,0.007843,0.050657,72.603375,72.689748
1,fgsm,0.015686,1,0.82,0.670,0.182927,0.860903,0.015686,0.046553,69.777397,69.858850
2,fgsm,0.031373,1,0.82,0.465,0.432927,1.709302,0.031373,0.038346,70.422693,70.496468
3,fgsm,0.062745,1,0.82,0.155,0.823171,3.423452,0.062745,0.046198,70.373601,70.455023
4,pgd,0.031373,20,0.82,0.550,0.335366,1.377881,0.031373,0.607778,70.256916,70.900174
5,deepfool,NaN,50,0.82,0.815,0.006098,0.021958,0.002786,17.533820,71.263038,88.833950
6,cw_l2,NaN,100,0.82,0.805,0.018293,0.099295,0.013628,5.894082,70.180672,76.110022


In [32]:
assert DIFFUSION_FINAL_METRICS_PATH.is_file()

verified_diffusion_df = pd.read_csv(
    DIFFUSION_FINAL_METRICS_PATH
)

assert list(
    verified_diffusion_df.columns
) == CANONICAL_METRIC_COLUMNS

assert len(verified_diffusion_df) == 7

assert verified_diffusion_df[
    "run_id"
].is_unique

assert verified_diffusion_df[
    "defense_id"
].eq("diff_purify").all()

assert verified_diffusion_df[
    "split"
].eq("diffusion_subset").all()

assert verified_diffusion_df[
    "num_samples"
].eq(200).all()

assert verified_diffusion_df[
    "clean_accuracy"
].eq(0.82).all()

assert set(
    verified_diffusion_df["attack_id"]
) == {
    "fgsm",
    "pgd",
    "deepfool",
    "cw_l2",
}

for column in (
    "clean_accuracy",
    "robust_accuracy",
    "attack_success_rate",
):
    assert verified_diffusion_df[
        column
    ].between(0.0, 1.0).all()


fgsm_rows = verified_diffusion_df.loc[
    verified_diffusion_df[
        "attack_id"
    ].eq("fgsm")
]

assert len(fgsm_rows) == 4

assert np.allclose(
    sorted(fgsm_rows["epsilon"].tolist()),
    sorted([
        2 / 255,
        4 / 255,
        8 / 255,
        16 / 255,
    ]),
)

pgd_rows = verified_diffusion_df.loc[
    verified_diffusion_df[
        "attack_id"
    ].eq("pgd")
]

assert len(pgd_rows) == 1

assert np.isclose(
    pgd_rows.iloc[0]["epsilon"],
    8 / 255,
)

assert np.isclose(
    pgd_rows.iloc[0]["alpha"],
    2 / 255,
)

unbounded_rows = verified_diffusion_df.loc[
    verified_diffusion_df[
        "attack_id"
    ].isin(["deepfool", "cw_l2"])
]

assert unbounded_rows["epsilon"].isna().all()
assert unbounded_rows["alpha"].isna().all()


for attack_config in DIFFUSION_ATTACK_CONFIGS:
    examples_path = diffusion_examples_path(
        attack_config["run_id"]
    )

    assert examples_path.is_file()

    examples = torch.load(
        examples_path,
        map_location="cpu",
        weights_only=False,
    )

    required_example_keys = {
        "originals",
        "adversarials",
        "purified_originals",
        "purified_adversarials",
        "labels",
        "clean_logits",
        "adversarial_logits",
    }

    assert required_example_keys.issubset(
        examples
    )

    assert (
        examples["originals"].shape
        == examples["adversarials"].shape
        == examples["purified_originals"].shape
        == examples[
            "purified_adversarials"
        ].shape
    )

    assert (
        examples["originals"].shape[0]
        == examples["labels"].shape[0]
    )

    assert examples[
        "purified_adversarials"
    ].min().item() >= 0.0

    assert examples[
        "purified_adversarials"
    ].max().item() <= 1.0

    print(
        "VERIFIED:",
        examples_path.name,
        "saved examples:",
        examples["labels"].shape[0],
        "total successful:",
        examples["successful_samples"],
    )


diffusion_metrics_hash = hashlib.sha256(
    DIFFUSION_FINAL_METRICS_PATH.read_bytes()
).hexdigest()

print(
    "\nDiffusion metrics SHA256:",
    diffusion_metrics_hash,
)

display(
    verified_diffusion_df[
        [
            "attack_id",
            "epsilon",
            "clean_accuracy",
            "robust_accuracy",
            "attack_success_rate",
            "mean_l2_successful",
            "mean_linf_successful",
            "attack_time_seconds",
            "defense_time_seconds",
            "total_time_seconds",
        ]
    ]
)

print(
    "\nALL FINAL DIFFUSION-PURIFICATION "
    "ATTACK EVALUATIONS PASSED."
)

VERIFIED: diff_purify__fgsm__eps2__t50__seed42__examples.pt saved examples: 8 total successful: 15
VERIFIED: diff_purify__fgsm__eps4__t50__seed42__examples.pt saved examples: 8 total successful: 30
VERIFIED: diff_purify__fgsm__eps8__t50__seed42__examples.pt saved examples: 8 total successful: 71
VERIFIED: diff_purify__fgsm__eps16__t50__seed42__examples.pt saved examples: 8 total successful: 135
VERIFIED: diff_purify__pgd__eps8__alpha2__steps20__t50__seed42__examples.pt saved examples: 8 total successful: 55
VERIFIED: diff_purify__deepfool__steps50__t50__seed42__examples.pt saved examples: 1 total successful: 1
VERIFIED: diff_purify__cw_l2__steps100__t50__seed42__examples.pt saved examples: 3 total successful: 3

Diffusion metrics SHA256: 72854353480c2734ec615aa243f8c038fe1e365bc3f7dde9b3d3774c69351482


,attack_id,epsilon,clean_accuracy,robust_accuracy,attack_success_rate,mean_l2_successful,mean_linf_successful,attack_time_seconds,defense_time_seconds,total_time_seconds
0,fgsm,0.007843,0.82,0.745,0.091463,0.433618,0.007843,0.050657,72.603375,72.689748
1,fgsm,0.015686,0.82,0.670,0.182927,0.860903,0.015686,0.046553,69.777397,69.858850
2,fgsm,0.031373,0.82,0.465,0.432927,1.709302,0.031373,0.038346,70.422693,70.496468
3,fgsm,0.062745,0.82,0.155,0.823171,3.423452,0.062745,0.046198,70.373601,70.455023
4,pgd,0.031373,0.82,0.550,0.335366,1.377881,0.031373,0.607778,70.256916,70.900174
5,deepfool,NaN,0.82,0.815,0.006098,0.021958,0.002786,17.533820,71.263038,88.833950
6,cw_l2,NaN,0.82,0.805,0.018293,0.099295,0.013628,5.894082,70.180672,76.110022



ALL FINAL DIFFUSION-PURIFICATION ATTACK EVALUATIONS PASSED.


In [33]:
from src.defenses.randomized_smoothing import (
    predict_smoothed,
)


FAIR_SUBSET_CLEAN_CACHE_PATH = (
    DIFFUSION_RESULTS_DIR
    / "fair_subset_clean_cache_seed42_n200.pt"
)


def fair_subset_synchronize():
    if DIFFUSION_DEVICE.type == "cuda":
        torch.cuda.synchronize(
            DIFFUSION_DEVICE
        )


def predict_classifier_in_batches(
    model,
    images,
    batch_size=128,
):
    model.eval()

    logit_batches = []

    fair_subset_synchronize()
    start_time = time.perf_counter()

    for start in range(
        0,
        images.shape[0],
        batch_size,
    ):
        batch = images[
            start:start + batch_size
        ].to(DIFFUSION_DEVICE)

        with torch.no_grad():
            logit_batches.append(
                model(batch).cpu()
            )

    fair_subset_synchronize()

    elapsed = (
        time.perf_counter() - start_time
    )

    logits = torch.cat(
        logit_batches,
        dim=0,
    )

    return logits, elapsed


if FAIR_SUBSET_CLEAN_CACHE_PATH.is_file():
    fair_subset_clean_cache = torch.load(
        FAIR_SUBSET_CLEAN_CACHE_PATH,
        map_location="cpu",
        weights_only=False,
    )

    assert torch.equal(
        torch.as_tensor(
            fair_subset_clean_cache[
                "subset_indices"
            ]
        ).long(),
        torch.tensor(
            DIFFUSION_SUBSET_INDICES,
            dtype=torch.long,
        ),
    )

    assert torch.equal(
        fair_subset_clean_cache[
            "labels"
        ].long(),
        DIFFUSION_LABELS,
    )

    print(
        "Existing fair-subset clean "
        "cache loaded."
    )

else:
    fair_predictions = {}
    fair_logits = {}
    fair_vote_counts = {}

    clean_accuracies = {}
    defense_times = {}
    classifier_times = {}

    # ----------------------------------------------
    # none و pgd_at
    # ----------------------------------------------

    for defense_id in (
        "none",
        "pgd_at",
    ):
        logits, classifier_time = (
            predict_classifier_in_batches(
                model=CLASSIFIERS[
                    defense_id
                ],
                images=DIFFUSION_RAW_IMAGES,
            )
        )

        predictions = logits.argmax(dim=1)

        fair_logits[defense_id] = logits
        fair_predictions[
            defense_id
        ] = predictions

        clean_accuracies[defense_id] = float(
            predictions.eq(
                DIFFUSION_LABELS
            ).float().mean().item()
        )

        defense_times[defense_id] = 0.0

        classifier_times[
            defense_id
        ] = classifier_time

    # ----------------------------------------------
    # randomized smoothing
    # ----------------------------------------------

    smoothing_model = CLASSIFIERS[
        "rand_smooth"
    ]

    smoothing_model.eval()

    smoothing_generator = torch.Generator(
        device=DIFFUSION_DEVICE
    ).manual_seed(DIFFUSION_SEED)

    smoothed_prediction_batches = []
    vote_count_batches = []

    fair_subset_synchronize()
    smoothing_start = time.perf_counter()

    for start in range(
        0,
        DIFFUSION_NUM_SAMPLES,
        32,
    ):
        images = DIFFUSION_RAW_IMAGES[
            start:start + 32
        ].to(DIFFUSION_DEVICE)

        (
            smoothed_predictions,
            vote_counts,
        ) = predict_smoothed(
            model=smoothing_model,
            images=images,
            sigma=0.25,
            num_samples=100,
            noise_batch_size=25,
            num_classes=10,
            clip_noise=True,
            generator=smoothing_generator,
        )

        smoothed_prediction_batches.append(
            smoothed_predictions.cpu()
        )

        vote_count_batches.append(
            vote_counts.cpu()
        )

    fair_subset_synchronize()

    smoothing_time = (
        time.perf_counter()
        - smoothing_start
    )

    rand_smooth_predictions = torch.cat(
        smoothed_prediction_batches,
        dim=0,
    )

    rand_smooth_vote_counts = torch.cat(
        vote_count_batches,
        dim=0,
    )

    assert torch.all(
        rand_smooth_vote_counts.sum(
            dim=1
        ) == 100
    )

    fair_predictions[
        "rand_smooth"
    ] = rand_smooth_predictions

    fair_vote_counts[
        "rand_smooth"
    ] = rand_smooth_vote_counts

    clean_accuracies[
        "rand_smooth"
    ] = float(
        rand_smooth_predictions.eq(
            DIFFUSION_LABELS
        ).float().mean().item()
    )

    defense_times[
        "rand_smooth"
    ] = smoothing_time

    classifier_times[
        "rand_smooth"
    ] = 0.0

    # ----------------------------------------------
    # ذخیرهٔ دائمی
    # ----------------------------------------------

    fair_subset_clean_cache = {
        "split": "diffusion_subset",
        "num_samples": (
            DIFFUSION_NUM_SAMPLES
        ),
        "seed": DIFFUSION_SEED,
        "subset_indices": torch.tensor(
            DIFFUSION_SUBSET_INDICES,
            dtype=torch.long,
        ),
        "labels": DIFFUSION_LABELS,
        "raw_images": (
            DIFFUSION_RAW_IMAGES
        ),
        "predictions": fair_predictions,
        "logits": fair_logits,
        "vote_counts": fair_vote_counts,
        "clean_accuracies": (
            clean_accuracies
        ),
        "defense_time_seconds": (
            defense_times
        ),
        "classifier_time_seconds": (
            classifier_times
        ),
        "randomized_smoothing": {
            "sigma": 0.25,
            "num_samples": 100,
            "noise_batch_size": 25,
            "clip_noise": True,
        },
    }

    temporary_cache_path = (
        FAIR_SUBSET_CLEAN_CACHE_PATH
        .with_name(
            FAIR_SUBSET_CLEAN_CACHE_PATH.name
            + ".tmp"
        )
    )

    torch.save(
        fair_subset_clean_cache,
        temporary_cache_path,
    )

    temporary_cache_path.replace(
        FAIR_SUBSET_CLEAN_CACHE_PATH
    )

    print(
        "Fair-subset clean cache created."
    )


fair_clean_accuracies = (
    fair_subset_clean_cache[
        "clean_accuracies"
    ]
)

fair_defense_times = (
    fair_subset_clean_cache[
        "defense_time_seconds"
    ]
)

for defense_id in (
    "none",
    "pgd_at",
    "rand_smooth",
):
    print(
        f"{defense_id}: "
        f"clean accuracy = "
        f"{100 * fair_clean_accuracies[defense_id]:.2f}%, "
        f"defense time = "
        f"{fair_defense_times[defense_id]:.2f}s"
    )


if (
    "rand_smooth"
    in fair_subset_clean_cache[
        "vote_counts"
    ]
):
    rand_votes = (
        fair_subset_clean_cache[
            "vote_counts"
        ]["rand_smooth"]
    )

    mean_voting_confidence = float(
        (
            rand_votes.max(dim=1).values
            / 100
        ).float().mean().item()
    )

    print(
        "Random-smoothing mean "
        "voting confidence:",
        f"{100 * mean_voting_confidence:.2f}%",
    )


fair_cache_hash = hashlib.sha256(
    FAIR_SUBSET_CLEAN_CACHE_PATH.read_bytes()
).hexdigest()

print("Saved:", FAIR_SUBSET_CLEAN_CACHE_PATH)
print("Cache SHA256:", fair_cache_hash)

print(
    "\nFAIR SUBSET CLEAN CACHE PASSED."
)

Fair-subset clean cache created.
none: clean accuracy = 87.50%, defense time = 0.00s
pgd_at: clean accuracy = 68.00%, defense time = 0.00s
rand_smooth: clean accuracy = 76.00%, defense time = 1.10s
Random-smoothing mean voting confidence: 81.38%
Saved: /content/drive/MyDrive/AI-Project-Adversarial-Attack-Defense/results/fair_subset_clean_cache_seed42_n200.pt
Cache SHA256: b9f0b9ea6170c7979ebe405584910da6e589834bc5804390b6ae9c83a6303efe

FAIR SUBSET CLEAN CACHE PASSED.
